# TATR Table Structure Recognition — LoRA Fine-Tuning

This notebook implements **LoRA fine-tuning** with **data augmentation** for Table Structure Recognition (TSR) using the vanilla Table Transformer model.

**Architecture:**
- Base: `microsoft/table-transformer-structure-recognition` (DETR-style, ResNet50 + Transformer)
- LoRA: r=16, α=32, dropout=0.05 via PEFT on encoder/decoder attention

**Pipeline:**
1. Install & configure environment
2. Load vanilla TATR model
3. Apply LoRA fine-tuning via PEFT library
4. Full data augmentation pipeline (15+ transforms from augmentation_config.json)
5. Runtime table cropping (15px padding) from raw images + in-notebook train/val/test split
6. Training loop with AdamW, cosine LR, gradient accumulation, AMP, early stopping
7. Two-way benchmark: (A) Vanilla TATR → (B) Vanilla TATR + LoRA + augmentation
8. Comprehensive evaluation and visualization

**Dataset:** 1,500 raw images with COCO annotations → runtime table cropping → image-level split (70% train / 15% val / 15% test)
**Categories:** table column (2), table row (3), table column header (4), projected row header (5), spanning cell (6)

## 1. Install Required Packages

In [ ]:
# Install required packages
!pip install -q transformers torch torchvision
!pip install -q pycocotools
!pip install -q matplotlib seaborn pandas pillow
!pip install -q tqdm
!pip install -q timm  # Required for table-transformer
!pip install -q peft  # LoRA fine-tuning
!pip install -q albumentations  # Data augmentation
!pip install -q opencv-python-headless  # Required by albumentations

## 2. Imports

In [ ]:
import os
import json
import random
import numpy as np
from typing import Dict, List, Tuple, Optional, Any
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from transformers import (
    DetrImageProcessor,
    TableTransformerForObjectDetection,
)

from peft import LoraConfig, get_peft_model

import albumentations as A
import cv2

from PIL import Image

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from tqdm.auto import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

import peft
print(f"PEFT version: {peft.__version__}")
import albumentations
print(f"Albumentations version: {albumentations.__version__}")

## 3. Configuration & Paths

In [ ]:
# =====================================================================
# Configuration — All hyperparameters in one place
# =====================================================================

# -------------------- PATHS --------------------
# Raw dataset (single COCO JSON + original images)
JSON_PATH = '/kaggle/input/datasets/mohammedahmedxx12/machathon-dataset/Phase1_Train_Dataset/annotations/Cells_Anotations_coco.json'
IMAGES_DIR = '/kaggle/input/datasets/mohammedahmedxx12/machathon-dataset/Phase1_Train_Dataset/images'

# Output directories
OUTPUT_DIR = '/kaggle/working'
CROPPED_DIR = os.path.join(OUTPUT_DIR, 'cropped_tables')
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')

# -------------------- MODEL --------------------
TSR_MODEL_NAME = "microsoft/table-transformer-structure-recognition"

# -------------------- TRAINING HYPERPARAMETERS --------------------
IMAGE_SIZE = 800  # Target image size (800x800)
BATCH_SIZE = 4  # Per-device batch size
GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch size = 4 * 4 = 16
LEARNING_RATE_LORA = 1e-3  # LR when LoRA is active
LEARNING_RATE_NO_LORA = 5e-5  # LR when LoRA is not active
WEIGHT_DECAY = 1e-5
MAX_GRAD_NORM = 0.01  # Gradient clipping
MAX_EPOCHS = 500
EARLY_STOPPING_PATIENCE = 10
USE_FP16 = True  # Mixed precision (only with LoRA)

# -------------------- LoRA CONFIGURATION --------------------
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_BIAS = "none"

# -------------------- INFERENCE / EVALUATION --------------------
CONFIDENCE_THRESHOLD = 0.5
TABLE_PADDING = 15  # Padding (in pixels) when cropping tables from raw images
MIN_CHILD_ANNOTATIONS = 3

# -------------------- DATASET SPLIT --------------------
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# -------------------- REPRODUCIBILITY --------------------
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Category IDs in the dataset:
# 1 = table (skip for TSR), 2 = table column, 3 = table row,
# 4 = table column header, 5 = table projected row header, 6 = table spanning cell
TABLE_CATEGORY_ID = 1
TSR_CATEGORY_IDS = [2, 3, 4, 5, 6]

# -------------------- PER-CATEGORY CONFIDENCE THRESHOLDS --------------------
PER_CATEGORY_CONFIDENCE_THRESHOLD = {
    2: 0.50,  # table column
    3: 0.57,  # table row
    4: 0.50,  # table column header
    5: 0.62,  # table projected row header
    6: 0.67,  # table spanning cell
}

# -------------------- CLASS-AWARE NMS SETTINGS --------------------
NMS_IOU_THRESHOLDS = {
    2: 0.30,  # table column
    3: 0.30,  # table row
    4: 0.45,  # table column header
    5: 0.40,  # table projected row header
    6: 0.35,  # table spanning cell
}

ENABLE_CLASS_AWARE_NMS = True
ENABLE_ROW_COVERAGE_CHECK = True
ENABLE_COLUMN_COUNT_CONSISTENCY = True
ENABLE_SPANNING_CELL_VALIDATION = True
ENABLE_HEADER_DEDUPLICATION = True

ROW_COVERAGE_MIN_RATIO = 0.85
COLUMN_COUNT_TOLERANCE = 0.15
MIN_SPANNING_CELLS_OVERLAP = 1.5
HIGH_CONFIDENCE_SPANNING_EXEMPTION = 0.75

# -------------------- SET SEEDS --------------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# -------------------- CREATE DIRECTORIES --------------------
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CROPPED_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# -------------------- PRINT CONFIG --------------------
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu_name}")

print(f"\n--- Model ---")
print(f"Model: {TSR_MODEL_NAME}")
print(f"Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")

print(f"\n--- Training ---")
print(f"Batch size: {BATCH_SIZE} (effective: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS})")
print(f"Learning rate (LoRA): {LEARNING_RATE_LORA}")
print(f"Max epochs: {MAX_EPOCHS}")
print(f"Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"Gradient clip: {MAX_GRAD_NORM}")
print(f"Mixed precision (fp16): {USE_FP16}")

print(f"\n--- LoRA ---")
print(f"Rank: {LORA_R}, Alpha: {LORA_ALPHA}, Dropout: {LORA_DROPOUT}")

print(f"\n--- Dataset ---")
print(f"Raw JSON: {JSON_PATH}")
print(f"Images: {IMAGES_DIR}")
print(f"Table padding: {TABLE_PADDING}px")
print(f"Split: {TRAIN_RATIO:.0%} train / {VAL_RATIO:.0%} val / {TEST_RATIO:.0%} test")

cat_name_map = {2: 'column', 3: 'row', 4: 'col header', 5: 'proj row hdr', 6: 'spanning cell'}
print(f"\n--- Per-Category Confidence Thresholds ---")
for cat_id, threshold in PER_CATEGORY_CONFIDENCE_THRESHOLD.items():
    print(f"  {cat_name_map[cat_id]}: {threshold:.2f}")

print(f"\n--- NMS IoU Thresholds ---")
for cat_id, threshold in NMS_IOU_THRESHOLDS.items():
    print(f"  {cat_name_map[cat_id]}: {threshold:.2f}")

## 4. Utility Functions (Cropping & Annotation Processing)

These functions handle raw image loading, table cropping, and annotation coordinate transformation.
They are preserved from the original inference pipeline and used for both baseline evaluation and test-time inference.

In [ ]:
def load_coco_json(json_path: str) -> dict:
    """
    Load COCO format JSON annotations.
    
    Args:
        json_path: Path to the JSON file
    
    Returns:
        Dictionary containing COCO data
    """
    if not os.path.exists(json_path):
        raise FileNotFoundError(f"JSON file not found at {json_path}")
    with open(json_path, 'r') as f:
        return json.load(f)


def count_child_annotations_per_image(coco_data: dict) -> Dict[int, int]:
    """
    Count non-table (child) annotations per image.
    
    Args:
        coco_data: Full COCO dataset
    
    Returns:
        Dictionary mapping image_id to count of child annotations
    """
    child_counts = defaultdict(int)
    for ann in coco_data['annotations']:
        if ann['category_id'] != TABLE_CATEGORY_ID:  # Not a table
            child_counts[ann['image_id']] += 1
    return child_counts


def filter_images_by_child_count(coco_data: dict, min_children: int = 3) -> dict:
    """
    Filter out images with <= min_children child (non-table) annotations.
    
    Args:
        coco_data: Full COCO dataset
        min_children: Minimum number of child annotations required (exclusive)
    
    Returns:
        Filtered COCO data
    """
    child_counts = count_child_annotations_per_image(coco_data)
    
    # Keep images with more than min_children child annotations
    valid_image_ids = set(
        img_id for img_id, count in child_counts.items() 
        if count > min_children
    )
    
    # Also include images that have child annotations (even if child_counts hasn't counted them)
    # This ensures we don't miss any images
    for img in coco_data['images']:
        if img['id'] not in child_counts:
            # Image has no child annotations, skip it
            pass
    
    filtered_images = [img for img in coco_data['images'] if img['id'] in valid_image_ids]
    filtered_annotations = [ann for ann in coco_data['annotations'] if ann['image_id'] in valid_image_ids]
    
    return {
        'images': filtered_images,
        'annotations': filtered_annotations,
        'categories': coco_data['categories']
    }


def get_table_annotations(coco_data: dict) -> Dict[int, List[dict]]:
    """
    Get all table annotations grouped by image_id.
    
    Args:
        coco_data: COCO dataset
    
    Returns:
        Dictionary mapping image_id to list of table annotations
    """
    table_anns = defaultdict(list)
    for ann in coco_data['annotations']:
        if ann['category_id'] == TABLE_CATEGORY_ID:
            table_anns[ann['image_id']].append(ann)
    return table_anns


def get_child_annotations_for_table(
    all_annotations: List[dict],
    table_bbox: List[float],
    tolerance: float = 0.0
) -> List[dict]:
    """
    Get child annotations that fall within a table's bounding box.
    
    Args:
        all_annotations: List of all annotations for an image
        table_bbox: Table bounding box [x, y, w, h]
        tolerance: Tolerance for checking if annotation is inside table
    
    Returns:
        List of child annotations within the table
    """
    tx, ty, tw, th = table_bbox
    tx2, ty2 = tx + tw, ty + th
    
    children = []
    for ann in all_annotations:
        if ann['category_id'] == TABLE_CATEGORY_ID:
            continue  # Skip table annotations
        
        ax, ay, aw, ah = ann['bbox']
        ax2, ay2 = ax + aw, ay + ah
        
        # Check if annotation center is within table bounds (with tolerance)
        center_x = ax + aw / 2
        center_y = ay + ah / 2
        
        if (tx - tolerance <= center_x <= tx2 + tolerance and
            ty - tolerance <= center_y <= ty2 + tolerance):
            children.append(ann)
    
    return children


def crop_table_with_padding(
    image: Image.Image,
    table_bbox: List[float],
    padding: int = 15
) -> Tuple[Image.Image, Tuple[int, int, int, int]]:
    """
    Crop a table from an image with padding (from the image itself, not whitespace).
    
    Args:
        image: PIL Image
        table_bbox: Table bounding box [x, y, w, h] in COCO format
        padding: Padding in pixels to add around the table
    
    Returns:
        Tuple of (cropped_image, actual_crop_coords)
    """
    img_w, img_h = image.size
    x, y, w, h = table_bbox
    
    # Calculate crop coordinates with padding, clamped to image bounds
    x1 = max(0, int(x - padding))
    y1 = max(0, int(y - padding))
    x2 = min(img_w, int(x + w + padding))
    y2 = min(img_h, int(y + h + padding))
    
    # Crop the image
    cropped = image.crop((x1, y1, x2, y2))
    
    return cropped, (x1, y1, x2, y2)


def transform_annotations_to_crop_coords(
    annotations: List[dict],
    crop_coords: Tuple[int, int, int, int]
) -> List[dict]:
    """
    Transform annotation coordinates relative to cropped image.
    
    Args:
        annotations: List of annotations with COCO format bboxes
        crop_coords: Crop coordinates (x1, y1, x2, y2)
    
    Returns:
        Annotations with transformed bboxes
    """
    x1_crop, y1_crop, x2_crop, y2_crop = crop_coords
    crop_w = x2_crop - x1_crop
    crop_h = y2_crop - y1_crop
    
    transformed = []
    for ann in annotations:
        ax, ay, aw, ah = ann['bbox']
        
        # Transform to crop coordinates
        new_x = ax - x1_crop
        new_y = ay - y1_crop
        
        # Clamp to crop bounds
        new_x = max(0, new_x)
        new_y = max(0, new_y)
        new_w = min(aw, crop_w - new_x)
        new_h = min(ah, crop_h - new_y)
        
        if new_w > 0 and new_h > 0:
            transformed.append({
                **ann,
                'bbox': [new_x, new_y, new_w, new_h],
                'original_bbox': ann['bbox']  # Keep original for reference
            })
    
    return transformed


print("Utility functions loaded.")

## 4b. Class-Aware NMS and Post-Processing Functions

This section implements:
- **Class-aware NMS**: Non-Maximum Suppression with per-category IoU thresholds
- **Row coverage check**: Ensure rows cover most of table height
- **Column count consistency**: All rows should span ~same number of columns
- **Spanning cell validation**: Spanning cells must overlap ≥2 rows or columns  
- **Header deduplication**: Only one column header region per table

In [ ]:
def compute_iou_single(box1: List[float], box2: List[float]) -> float:
    """
    Compute IoU between two boxes in COCO format [x, y, w, h].
    
    Args:
        box1, box2: Boxes in [x, y, width, height] format
    
    Returns:
        IoU score (0.0 to 1.0)
    """
    # Convert to [x1, y1, x2, y2]
    x1_1, y1_1 = box1[0], box1[1]
    x2_1, y2_1 = box1[0] + box1[2], box1[1] + box1[3]
    
    x1_2, y1_2 = box2[0], box2[1]
    x2_2, y2_2 = box2[0] + box2[2], box2[1] + box2[3]
    
    # Compute intersection
    x1_i = max(x1_1, x1_2)
    y1_i = max(y1_1, y1_2)
    x2_i = min(x2_1, x2_2)
    y2_i = min(y2_1, y2_2)
    
    if x2_i <= x1_i or y2_i <= y1_i:
        return 0.0
    
    intersection = (x2_i - x1_i) * (y2_i - y1_i)
    
    # Compute union
    area1 = box1[2] * box1[3]
    area2 = box2[2] * box2[3]
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0.0


def class_aware_nms(
    boxes: List[List[float]],
    scores: List[float],
    labels: List[int],
    nms_thresholds: Dict[int, float],
    default_threshold: float = 0.5
) -> Tuple[List[List[float]], List[float], List[int]]:
    """
    Apply Non-Maximum Suppression per category with category-specific IoU thresholds.
    
    Args:
        boxes: List of boxes in COCO format [x, y, w, h]
        scores: Confidence scores for each box
        labels: Category labels for each box
        nms_thresholds: Dict mapping category_id to NMS IoU threshold
        default_threshold: Fallback IoU threshold if category not in nms_thresholds
    
    Returns:
        Tuple of (filtered_boxes, filtered_scores, filtered_labels)
    """
    if len(boxes) == 0:
        return [], [], []
    
    # Group indices by category
    category_indices = defaultdict(list)
    for idx, label in enumerate(labels):
        category_indices[label].append(idx)
    
    # Apply NMS per category
    keep_indices = []
    
    for cat_id, indices in category_indices.items():
        if len(indices) == 0:
            continue
        
        # Get NMS threshold for this category
        iou_threshold = nms_thresholds.get(cat_id, default_threshold)
        
        # Get boxes and scores for this category
        cat_boxes = [boxes[i] for i in indices]
        cat_scores = [scores[i] for i in indices]
        
        # Sort by score (descending)
        sorted_idx = np.argsort(cat_scores)[::-1]
        
        keep_cat = []
        suppressed = set()
        
        for i in sorted_idx:
            if i in suppressed:
                continue
            
            keep_cat.append(indices[i])
            
            # Suppress overlapping boxes
            for j in sorted_idx:
                if j in suppressed or j == i:
                    continue
                
                iou = compute_iou_single(cat_boxes[i], cat_boxes[j])
                if iou >= iou_threshold:
                    suppressed.add(j)
        
        keep_indices.extend(keep_cat)
    
    # Sort to maintain order by score
    keep_indices = sorted(keep_indices, key=lambda i: scores[i], reverse=True)
    
    filtered_boxes = [boxes[i] for i in keep_indices]
    filtered_scores = [scores[i] for i in keep_indices]
    filtered_labels = [labels[i] for i in keep_indices]
    
    return filtered_boxes, filtered_scores, filtered_labels


def apply_per_category_confidence_threshold(
    boxes: List[List[float]],
    scores: List[float],
    labels: List[int],
    thresholds: Dict[int, float],
    default_threshold: float = 0.5
) -> Tuple[List[List[float]], List[float], List[int]]:
    """
    Filter predictions using per-category confidence thresholds.
    
    Args:
        boxes: List of boxes in COCO format [x, y, w, h]
        scores: Confidence scores for each box
        labels: Category labels for each box
        thresholds: Dict mapping category_id to confidence threshold
        default_threshold: Fallback threshold if category not in thresholds
    
    Returns:
        Tuple of (filtered_boxes, filtered_scores, filtered_labels)
    """
    if len(boxes) == 0:
        return [], [], []
    
    filtered_boxes = []
    filtered_scores = []
    filtered_labels = []
    
    for box, score, label in zip(boxes, scores, labels):
        threshold = thresholds.get(label, default_threshold)
        if score >= threshold:
            filtered_boxes.append(box)
            filtered_scores.append(score)
            filtered_labels.append(label)
    
    return filtered_boxes, filtered_scores, filtered_labels


print("Class-aware NMS functions loaded.")

In [ ]:
def check_row_coverage(
    row_boxes: List[List[float]],
    table_height: float,
    min_coverage_ratio: float = 0.85
) -> Tuple[bool, float, List[Tuple[float, float]]]:
    """
    Check if rows collectively cover most of the table height.
    Identifies gaps where rows might need to be split.
    
    Args:
        row_boxes: List of row boxes in COCO format [x, y, w, h]
        table_height: Height of the table/cropped image
        min_coverage_ratio: Minimum fraction of table height that should be covered
    
    Returns:
        Tuple of (is_coverage_sufficient, coverage_ratio, gaps)
        gaps: List of (gap_start_y, gap_end_y) tuples for regions not covered
    """
    if len(row_boxes) == 0:
        return False, 0.0, [(0, table_height)]
    
    # Sort rows by y-coordinate (top to bottom)
    sorted_rows = sorted(row_boxes, key=lambda b: b[1])
    
    # Compute coverage intervals
    covered_intervals = []
    for box in sorted_rows:
        y1 = box[1]
        y2 = box[1] + box[3]
        covered_intervals.append((y1, y2))
    
    # Merge overlapping intervals
    merged_intervals = []
    for start, end in covered_intervals:
        if merged_intervals and start <= merged_intervals[-1][1]:
            # Overlapping - extend the previous interval
            merged_intervals[-1] = (merged_intervals[-1][0], max(merged_intervals[-1][1], end))
        else:
            merged_intervals.append((start, end))
    
    # Calculate total covered height
    total_covered = sum(end - start for start, end in merged_intervals)
    coverage_ratio = total_covered / table_height if table_height > 0 else 0.0
    
    # Find gaps
    gaps = []
    prev_end = 0
    for start, end in merged_intervals:
        if start > prev_end + 5:  # Allow 5px tolerance for small gaps
            gaps.append((prev_end, start))
        prev_end = end
    
    # Check gap at the end
    if table_height - prev_end > 5:
        gaps.append((prev_end, table_height))
    
    is_sufficient = coverage_ratio >= min_coverage_ratio
    
    return is_sufficient, coverage_ratio, gaps


def check_column_count_consistency(
    row_boxes: List[List[float]],
    column_boxes: List[List[float]],
    tolerance: float = 0.15
) -> Tuple[bool, Dict[int, int]]:
    """
    Check that all rows span roughly the same number of columns.
    
    Args:
        row_boxes: List of row boxes in COCO format [x, y, w, h]
        column_boxes: List of column boxes in COCO format [x, y, w, h]
        tolerance: Allowed variation ratio in column count
    
    Returns:
        Tuple of (is_consistent, row_to_column_count_dict)
    """
    if len(row_boxes) == 0 or len(column_boxes) == 0:
        return True, {}
    
    # For each row, count how many columns it intersects
    row_column_counts = {}
    
    for row_idx, row_box in enumerate(row_boxes):
        rx1, ry1 = row_box[0], row_box[1]
        rx2, ry2 = rx1 + row_box[2], ry1 + row_box[3]
        row_center_y = ry1 + row_box[3] / 2
        
        col_count = 0
        for col_box in column_boxes:
            cx1, cy1 = col_box[0], col_box[1]
            cx2, cy2 = cx1 + col_box[2], cy1 + col_box[3]
            
            # Check if row and column overlap horizontally
            # and if row center is within column's vertical span
            if rx1 < cx2 and rx2 > cx1:  # Horizontal overlap
                col_count += 1
        
        row_column_counts[row_idx] = col_count
    
    if len(row_column_counts) == 0:
        return True, row_column_counts
    
    # Check consistency
    counts = list(row_column_counts.values())
    if len(counts) == 0:
        return True, row_column_counts
    
    median_count = np.median(counts)
    if median_count == 0:
        return True, row_column_counts
    
    # Check if all counts are within tolerance of median
    is_consistent = all(
        abs(c - median_count) / median_count <= tolerance
        for c in counts
    )
    
    return is_consistent, row_column_counts


def validate_spanning_cells(
    spanning_boxes: List[List[float]],
    spanning_scores: List[float],
    row_boxes: List[List[float]],
    column_boxes: List[List[float]],
    min_overlap: float = 1.5,
    high_confidence_exemption: float = 0.75
) -> Tuple[List[List[float]], List[float], List[int]]:
    """
    Validate spanning cells: each must overlap at least min_overlap rows OR columns.
    Spanning cells with confidence > high_confidence_exemption bypass validation.
    
    Args:
        spanning_boxes: List of spanning cell boxes in COCO format [x, y, w, h]
        spanning_scores: Confidence scores for spanning cells
        row_boxes: List of row boxes
        column_boxes: List of column boxes
        min_overlap: Minimum number of rows OR columns a spanning cell must overlap (allows fractional)
        high_confidence_exemption: Confidence threshold to bypass validation
    
    Returns:
        Tuple of (valid_boxes, valid_scores, indices_removed)
    """
    if len(spanning_boxes) == 0:
        return [], [], []
    
    valid_boxes = []
    valid_scores = []
    removed_indices = []
    
    for idx, (span_box, span_score) in enumerate(zip(spanning_boxes, spanning_scores)):
        # High confidence exemption -> keep unconditionally
        if span_score >= high_confidence_exemption:
            valid_boxes.append(span_box)
            valid_scores.append(span_score)
            continue
            
        sx1, sy1 = span_box[0], span_box[1]
        sx2, sy2 = sx1 + span_box[2], sy1 + span_box[3]
        
        # Calculate fractional row overlaps
        row_overlap_sum = 0.0
        for row_box in row_boxes:
            ry1, ry2 = row_box[1], row_box[1] + row_box[3]
            overlap = max(0, min(sy2, ry2) - max(sy1, ry1))
            if overlap > 0 and row_box[3] > 0:
                # Add fractional overlap capped at 1.0 per row
                row_overlap_sum += min(1.0, overlap / row_box[3])
                
        # Calculate fractional column overlaps
        col_overlap_sum = 0.0
        for col_box in column_boxes:
            cx1, cx2 = col_box[0], col_box[0] + col_box[2]
            overlap = max(0, min(sx2, cx2) - max(sx1, cx1))
            if overlap > 0 and col_box[2] > 0:
                # Add fractional overlap capped at 1.0 per column
                col_overlap_sum += min(1.0, overlap / col_box[2])
        
        # Valid if spans >= min_overlap rows OR >= min_overlap columns
        is_valid = row_overlap_sum >= min_overlap or col_overlap_sum >= min_overlap
        
        if is_valid:
            valid_boxes.append(span_box)
            valid_scores.append(span_score)
        else:
            removed_indices.append(idx)
    
    return valid_boxes, valid_scores, removed_indices


def deduplicate_headers(
    header_boxes: List[List[float]],
    header_scores: List[float],
    max_headers: int = 1
) -> Tuple[List[List[float]], List[float]]:
    """
    Ensure only one column header region per table.
    Keep the header with highest confidence score.
    
    Args:
        header_boxes: List of header boxes in COCO format [x, y, w, h]
        header_scores: Confidence scores for headers
        max_headers: Maximum number of headers to keep
    
    Returns:
        Tuple of (filtered_boxes, filtered_scores)
    """
    if len(header_boxes) <= max_headers:
        return header_boxes, header_scores
    
    # Sort by score and keep top max_headers
    sorted_indices = np.argsort(header_scores)[::-1][:max_headers]
    
    filtered_boxes = [header_boxes[i] for i in sorted_indices]
    filtered_scores = [header_scores[i] for i in sorted_indices]
    
    return filtered_boxes, filtered_scores


print("Post-processing validation functions loaded.")

In [ ]:
def apply_table_structure_postprocessing(
    pred_boxes: List[List[float]],
    pred_scores: List[float],
    pred_labels: List[int],
    image_size: Tuple[int, int],
    config: Dict[str, Any] = None
) -> Tuple[List[List[float]], List[float], List[int], Dict[str, Any]]:
    """
    Apply complete post-processing pipeline for table structure predictions.
    
    Pipeline steps:
    1. Apply per-category confidence thresholds
    2. Apply class-aware NMS
    3. Validate spanning cells
    4. Deduplicate headers
    5. Check row/column consistency (for diagnostics)
    
    Args:
        pred_boxes: Predicted boxes in COCO format [x, y, w, h]
        pred_scores: Confidence scores
        pred_labels: Category labels (2=column, 3=row, 4=header, 5=proj_hdr, 6=spanning)
        image_size: (width, height) of the cropped table image
        config: Configuration dict with thresholds and flags
    
    Returns:
        Tuple of (filtered_boxes, filtered_scores, filtered_labels, diagnostics)
    """
    if config is None:
        config = {
            'confidence_thresholds': PER_CATEGORY_CONFIDENCE_THRESHOLD,
            'nms_thresholds': NMS_IOU_THRESHOLDS,
            'enable_nms': ENABLE_CLASS_AWARE_NMS,
            'enable_row_coverage': ENABLE_ROW_COVERAGE_CHECK,
            'enable_column_consistency': ENABLE_COLUMN_COUNT_CONSISTENCY,
            'enable_spanning_validation': ENABLE_SPANNING_CELL_VALIDATION,
            'enable_header_dedup': ENABLE_HEADER_DEDUPLICATION,
            'min_spanning_overlap': MIN_SPANNING_CELLS_OVERLAP,
            'high_confidence_exemption': HIGH_CONFIDENCE_SPANNING_EXEMPTION,
            'row_coverage_ratio': ROW_COVERAGE_MIN_RATIO,
        }
    
    diagnostics = {
        'original_count': len(pred_boxes),
        'removed_by_confidence': 0,
        'removed_by_nms': 0,
        'removed_spanning_cells': 0,
        'removed_headers': 0,
        'row_coverage_ok': True,
        'column_consistency_ok': True,
    }
    
    img_width, img_height = image_size
    
    # Step 1: Apply per-category confidence thresholds
    filtered_boxes, filtered_scores, filtered_labels = apply_per_category_confidence_threshold(
        pred_boxes, pred_scores, pred_labels,
        config['confidence_thresholds'],
        default_threshold=0.5
    )
    diagnostics['removed_by_confidence'] = len(pred_boxes) - len(filtered_boxes)
    
    if len(filtered_boxes) == 0:
        return [], [], [], diagnostics
    
    # Step 2: Apply class-aware NMS
    if config['enable_nms']:
        pre_nms_count = len(filtered_boxes)
        filtered_boxes, filtered_scores, filtered_labels = class_aware_nms(
            filtered_boxes, filtered_scores, filtered_labels,
            config['nms_thresholds'],
            default_threshold=0.5
        )
        diagnostics['removed_by_nms'] = pre_nms_count - len(filtered_boxes)
    
    if len(filtered_boxes) == 0:
        return [], [], [], diagnostics
    
    # Separate by category for further processing
    row_idx = [i for i, l in enumerate(filtered_labels) if l == 3]
    col_idx = [i for i, l in enumerate(filtered_labels) if l == 2]
    header_idx = [i for i, l in enumerate(filtered_labels) if l == 4]
    spanning_idx = [i for i, l in enumerate(filtered_labels) if l == 6]
    other_idx = [i for i, l in enumerate(filtered_labels) if l not in [2, 3, 4, 6]]
    
    row_boxes = [filtered_boxes[i] for i in row_idx]
    col_boxes = [filtered_boxes[i] for i in col_idx]
    header_boxes = [filtered_boxes[i] for i in header_idx]
    header_scores = [filtered_scores[i] for i in header_idx]
    spanning_boxes = [filtered_boxes[i] for i in spanning_idx]
    spanning_scores = [filtered_scores[i] for i in spanning_idx]
    
    # Step 3: Validate spanning cells
    if config['enable_spanning_validation'] and len(spanning_boxes) > 0:
        valid_spanning, valid_spanning_scores, removed_idx = validate_spanning_cells(
            spanning_boxes, spanning_scores,
            row_boxes, col_boxes,
            min_overlap=config.get('min_spanning_overlap', 1.5),
            high_confidence_exemption=config.get('high_confidence_exemption', 0.75)
        )
        diagnostics['removed_spanning_cells'] = len(removed_idx)
    else:
        valid_spanning = spanning_boxes
        valid_spanning_scores = spanning_scores
    
    # Step 4: Deduplicate headers
    if config['enable_header_dedup'] and len(header_boxes) > 1:
        valid_headers, valid_header_scores = deduplicate_headers(
            header_boxes, header_scores, max_headers=1
        )
        diagnostics['removed_headers'] = len(header_boxes) - len(valid_headers)
    else:
        valid_headers = header_boxes
        valid_header_scores = header_scores
    
    # Step 5: Check row coverage (diagnostic only, doesn't filter)
    if config['enable_row_coverage'] and len(row_boxes) > 0:
        is_sufficient, coverage_ratio, gaps = check_row_coverage(
            row_boxes, img_height, config['row_coverage_ratio']
        )
        diagnostics['row_coverage_ok'] = is_sufficient
        diagnostics['row_coverage_ratio'] = coverage_ratio
        diagnostics['row_gaps'] = gaps
    
    # Step 6: Check column consistency (diagnostic only)
    if config['enable_column_consistency'] and len(row_boxes) > 0 and len(col_boxes) > 0:
        is_consistent, row_col_counts = check_column_count_consistency(
            row_boxes, col_boxes,
            tolerance=COLUMN_COUNT_TOLERANCE
        )
        diagnostics['column_consistency_ok'] = is_consistent
        diagnostics['row_column_counts'] = row_col_counts
    
    # Reassemble final predictions
    final_boxes = []
    final_scores = []
    final_labels = []
    
    # Add back all categories with filtering applied
    for i in other_idx:  # Projected row headers and others
        final_boxes.append(filtered_boxes[i])
        final_scores.append(filtered_scores[i])
        final_labels.append(filtered_labels[i])
    
    for i in row_idx:  # Rows (unchanged)
        final_boxes.append(filtered_boxes[i])
        final_scores.append(filtered_scores[i])
        final_labels.append(filtered_labels[i])
    
    for i in col_idx:  # Columns (unchanged)
        final_boxes.append(filtered_boxes[i])
        final_scores.append(filtered_scores[i])
        final_labels.append(filtered_labels[i])
    
    # Add validated headers
    for box, score in zip(valid_headers, valid_header_scores):
        final_boxes.append(box)
        final_scores.append(score)
        final_labels.append(4)
    
    # Add validated spanning cells
    for box, score in zip(valid_spanning, valid_spanning_scores):
        final_boxes.append(box)
        final_scores.append(score)
        final_labels.append(6)
    
    diagnostics['final_count'] = len(final_boxes)
    
    return final_boxes, final_scores, final_labels, diagnostics


print("Complete post-processing pipeline loaded.")

## 5. LoRA Configuration & Application

Implementation of LoRA fine-tuning for the vanilla TATR model:
- **get_lora_target_modules()**: Returns target modules for LoRA injection
- **apply_lora_to_model()**: Applies LoRA via PEFT to transformer attention layers

In [ ]:
# =====================================================================
# LoRA Target Modules for Vanilla TATR
# =====================================================================

def get_lora_target_modules(model: nn.Module) -> List[str]:
    """
    Return the list of LoRA target modules for vanilla TATR model.
    
    Targets:
    - Encoder self-attention q/k/v/out_proj (layers 0-5)
    - Decoder self-attention and cross-attention projections (layers 0-5)
    """
    target_modules = []
    
    # --- Transformer Encoder self-attention (layers 0-5) ---
    for layer_idx in range(6):
        for proj in ["q_proj", "k_proj", "v_proj", "out_proj"]:
            target_modules.append(
                f"model.encoder.layers.{layer_idx}.self_attn.{proj}"
            )
    
    # --- Transformer Decoder self-attention and cross-attention (layers 0-5) ---
    for layer_idx in range(6):
        # Self-attention
        for proj in ["q_proj", "k_proj", "v_proj", "out_proj"]:
            target_modules.append(
                f"model.decoder.layers.{layer_idx}.self_attn.{proj}"
            )
        # Cross-attention (encoder_attn)
        for proj in ["q_proj", "k_proj", "v_proj", "out_proj"]:
            target_modules.append(
                f"model.decoder.layers.{layer_idx}.encoder_attn.{proj}"
            )
    
    # Validate: check which modules actually exist in the model
    existing_names = {name for name, _ in model.named_modules()}
    valid_targets = []
    missing_targets = []
    
    for target in target_modules:
        # Check if target exists or any module name starts with it
        found = any(name == target or name.startswith(target + ".") for name in existing_names)
        if found:
            valid_targets.append(target)
        else:
            missing_targets.append(target)
    
    if missing_targets:
        print(f"  [LoRA] Warning: {len(missing_targets)} target modules not found:")
        for m in missing_targets[:10]:
            print(f"    - {m}")
    
    print(f"  [LoRA] {len(valid_targets)}/{len(target_modules)} target modules validated")
    return valid_targets


def apply_lora_to_model(
    model: TableTransformerForObjectDetection,
    target_modules: List[str],
) -> nn.Module:
    """
    Apply LoRA fine-tuning to the model using PEFT.
    """
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=target_modules,
        lora_dropout=LORA_DROPOUT,
        bias=LORA_BIAS,
    )
    
    peft_model = get_peft_model(model, lora_config)
    
    # Print parameter statistics
    trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in peft_model.parameters())
    
    print(f"\n  [LoRA] Applied successfully!")
    print(f"  [LoRA] Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
    print(f"  [LoRA] Total parameters: {total_params:,}")
    print(f"  [LoRA] Config: r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
    
    return peft_model


print("LoRA configuration functions defined.")

In [ ]:
# =====================================================================
# Data Augmentation Pipeline
# From augmentation_config.json + Architecture_Modules_and_Configurations.md
# + Tablecert YOLO and TATR Enhanced Models paper
# =====================================================================

def build_train_augmentation() -> A.Compose:
    """
    Build the full training augmentation pipeline.
    
    All transforms are derived from:
      - Architecture_Modules_and_Configurations.md §3.4
      - Tablecert paper: perspective distortion, horizontal flip, brightness/contrast, HSV
    
    Applied ONLY during training, AFTER table cropping, BEFORE model input.
    Uses albumentations BboxParams to preserve annotation integrity.
    """
    transforms = []
    
    # --- Geometric transforms ---
    # Horizontal flip (p=0.3) — safe for tables, re-anchors bbox x coordinates
    transforms.append(A.HorizontalFlip(p=0.3))
    
    # Rotation ±5° (p=0.2) — simulates scanning errors
    transforms.append(A.Rotate(
        limit=5,
        p=0.2,
        border_mode=cv2.BORDER_CONSTANT,
        value=(255, 255, 255),  # White border for document images
    ))
    
    # Perspective distortion (p=0.15) — camera/scanner artifacts
    transforms.append(A.Perspective(
        scale=(0.02, 0.05),
        p=0.15,
    ))
    
    # Affine transforms (p=0.15) — layout variation
    transforms.append(A.Affine(
        scale=(0.95, 1.05),
        translate_percent={'x': (-0.02, 0.02), 'y': (-0.02, 0.02)},
        shear=(-2, 2),
        p=0.15,
        border_mode=cv2.BORDER_CONSTANT,
        cval=(255, 255, 255),
    ))
    
    # --- Photometric transforms ---
    # Brightness/contrast (p=0.4) — varying document quality
    transforms.append(A.RandomBrightnessContrast(
        brightness_limit=0.15,
        contrast_limit=0.15,
        p=0.4,
    ))
    
    # HSV shift (p=0.3) — ink/paper color variation
    transforms.append(A.HueSaturationValue(
        hue_shift_limit=10,
        sat_shift_limit=15,
        val_shift_limit=15,
        p=0.3,
    ))
    
    # Grayscale conversion (p=0.1) — some docs are B&W
    transforms.append(A.ToGray(p=0.1))
    
    # Gaussian blur (p=0.2) — low-quality scans
    transforms.append(A.GaussianBlur(
        blur_limit=(3, 3),
        p=0.2,
    ))
    
    # Gaussian noise (p=0.15) — scanner artifacts
    transforms.append(A.GaussNoise(
        var_limit=(5.0, 20.0),
        p=0.15,
    ))
    
    # ISO noise (p=0.1) — camera capture noise
    transforms.append(A.ISONoise(
        intensity=(0.1, 0.3),
        p=0.1,
    ))
    
    # JPEG compression artifacts (p=0.2)
    transforms.append(A.ImageCompression(
        quality_lower=50,
        quality_upper=95,
        p=0.2,
    ))
    
    # --- Document-specific transforms ---
    # Random shadow (p=0.1) — page shadows/folds
    transforms.append(A.RandomShadow(
        num_shadows_limit=(1, 2),
        shadow_dimension=5,
        p=0.1,
    ))
    
    # Grid distortion (p=0.1) — page warping
    transforms.append(A.GridDistortion(
        distort_limit=0.1,
        p=0.1,
        border_mode=cv2.BORDER_CONSTANT,
        value=(255, 255, 255),
    ))
    
    # Compose with bbox-safe params
    augmentation = A.Compose(
        transforms,
        bbox_params=A.BboxParams(
            format='coco',  # [x_min, y_min, width, height]
            min_area=100,
            min_visibility=0.3,
            label_fields=['category_ids'],
        ),
    )
    
    return augmentation


# Build augmentation
train_augmentation = build_train_augmentation()

print(f"Training augmentation pipeline built with {len(train_augmentation.transforms)} transforms:")
for i, t in enumerate(train_augmentation.transforms):
    name = t.__class__.__name__
    p = getattr(t, 'p', 'N/A')
    print(f"  {i+1:2d}. {name} (p={p})")
print(f"\nBbox params: format=coco, min_area=100, min_visibility=0.3")
print(f"Augmentation applied: ONLY during training, AFTER table cropping")

## 6. Data Pipeline — Raw Dataset with Runtime Table Cropping

Loads the single COCO JSON annotation file, finds all table annotations (category_id=1), crops each table from its source image with 15px padding, collects child structure annotations (categories 2–6), and transforms their coordinates into the cropped frame.

Train/val/test split is performed at the **image level** (70/15/15) to prevent data leakage (tables from the same image always stay in the same split).

**Pipeline:** Raw image → Crop table region (15px padding) → Transform child annotations → Apply augmentation (train only) → DETR Processor

In [ ]:
# =====================================================================
# Build per-table crop manifest from raw COCO JSON
# =====================================================================

def build_crop_manifest(coco_data: dict) -> List[dict]:
    """
    Build a list of crop entries — one per table in the dataset.
    
    Each entry stores everything needed to produce a single training sample:
      - source image file name
      - table bbox (for cropping with padding)
      - child (structure) annotations inside that table
    
    Tables whose child count <= MIN_CHILD_ANNOTATIONS are skipped.
    
    Returns:
        List of dicts, each with keys:
            image_id, file_name, table_bbox, child_annotations
    """
    # Index images
    id_to_img = {img['id']: img for img in coco_data['images']}
    
    # Index all annotations by image_id
    img_to_anns = defaultdict(list)
    for ann in coco_data['annotations']:
        img_to_anns[ann['image_id']].append(ann)
    
    # Get table annotations per image
    table_anns = get_table_annotations(coco_data)  # from utility cell
    
    manifest = []
    skipped_low_children = 0
    
    for image_id, tables in table_anns.items():
        img_info = id_to_img.get(image_id)
        if img_info is None:
            continue
        
        all_anns_for_image = img_to_anns[image_id]
        
        for table_ann in tables:
            table_bbox = table_ann['bbox']  # [x, y, w, h]
            
            # Get child (structure) annotations inside this table
            children = get_child_annotations_for_table(
                all_anns_for_image, table_bbox, tolerance=5.0
            )
            
            if len(children) <= MIN_CHILD_ANNOTATIONS:
                skipped_low_children += 1
                continue
            
            manifest.append({
                'image_id': image_id,
                'file_name': img_info['file_name'],
                'table_bbox': table_bbox,
                'table_ann_id': table_ann['id'],
                'child_annotations': children,
            })
    
    print(f"  Built crop manifest: {len(manifest)} table crops "
          f"(skipped {skipped_low_children} tables with <= {MIN_CHILD_ANNOTATIONS} children)")
    return manifest


def split_manifest_by_image(
    manifest: List[dict],
    train_ratio: float,
    val_ratio: float,
    test_ratio: float,
    seed: int = 42,
) -> Tuple[List[dict], List[dict], List[dict]]:
    """
    Split manifest into train/val/test at the IMAGE level.
    
    All tables from the same image are kept in the same split
    to prevent data leakage.
    
    Returns:
        (train_manifest, val_manifest, test_manifest)
    """
    # Group crops by image_id
    img_to_crops = defaultdict(list)
    for entry in manifest:
        img_to_crops[entry['image_id']].append(entry)
    
    # Shuffle image IDs
    image_ids = sorted(img_to_crops.keys())
    rng = random.Random(seed)
    rng.shuffle(image_ids)
    
    n = len(image_ids)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    # n_test = remainder
    
    train_ids = set(image_ids[:n_train])
    val_ids = set(image_ids[n_train:n_train + n_val])
    test_ids = set(image_ids[n_train + n_val:])
    
    train_manifest = [e for e in manifest if e['image_id'] in train_ids]
    val_manifest = [e for e in manifest if e['image_id'] in val_ids]
    test_manifest = [e for e in manifest if e['image_id'] in test_ids]
    
    print(f"  Split (image-level): {len(train_ids)} train imgs → {len(train_manifest)} crops, "
          f"{len(val_ids)} val imgs → {len(val_manifest)} crops, "
          f"{len(test_ids)} test imgs → {len(test_manifest)} crops")
    
    return train_manifest, val_manifest, test_manifest


# =====================================================================
# TSR Training/Evaluation Dataset — Raw Images + Runtime Cropping
# =====================================================================

class TSRTrainDataset(Dataset):
    """
    Dataset for TATR training/evaluation using raw images.
    
    For each sample (one table crop):
      1. Load the source image from disk
      2. Crop the table region with TABLE_PADDING (15px)
      3. Collect child annotations and transform to crop coordinates
      4. Apply augmentation (train only)
      5. Run through DETR processor
    
    Args:
        crop_manifest: List of crop entries from build_crop_manifest()
        images_dir: Path to raw source images
        processor: DetrImageProcessor instance
        augmentation: Optional albumentations Compose (train only)
        mode: 'train', 'val', or 'test'
    """
    
    def __init__(
        self,
        crop_manifest: List[dict],
        images_dir: str,
        processor,
        augmentation: Optional[A.Compose] = None,
        mode: str = 'train',
    ):
        self.manifest = crop_manifest
        self.images_dir = images_dir
        self.processor = processor
        self.augmentation = augmentation if mode == 'train' else None
        self.mode = mode
        
        total_anns = sum(len(e['child_annotations']) for e in self.manifest)
        print(f"  TSRTrainDataset [{mode}]: {len(self.manifest)} crops, "
              f"{total_anns} child annotations")
    
    def __len__(self):
        return len(self.manifest)
    
    def __getitem__(self, idx):
        entry = self.manifest[idx]
        
        # --- Load source image ---
        img_path = os.path.join(self.images_dir, entry['file_name'])
        
        if os.path.exists(img_path):
            source_image = Image.open(img_path).convert('RGB')
        else:
            # Fallback: white placeholder (should not happen on Kaggle)
            source_image = Image.new('RGB', (800, 600), (255, 255, 255))
        
        # --- Crop table with 15px padding ---
        cropped, crop_coords = crop_table_with_padding(
            source_image, entry['table_bbox'], padding=TABLE_PADDING
        )
        
        # --- Transform child annotations to crop coordinates ---
        transformed_anns = transform_annotations_to_crop_coords(
            entry['child_annotations'], crop_coords
        )
        
        # --- Prepare annotations (COCO format) ---
        bboxes = []
        category_ids = []
        for ann in transformed_anns:
            x, y, w, h = ann['bbox']
            if w > 0 and h > 0:
                bboxes.append([x, y, w, h])
                category_ids.append(ann['category_id'])
        
        # --- Apply augmentation (train only) ---
        image_np = np.array(cropped)
        
        if self.augmentation is not None and len(bboxes) > 0:
            try:
                augmented = self.augmentation(
                    image=image_np,
                    bboxes=bboxes,
                    category_ids=category_ids,
                )
                image_np = augmented['image']
                bboxes = augmented['bboxes']
                category_ids = augmented['category_ids']
            except Exception:
                # If augmentation fails (e.g., all boxes removed), use original
                pass
        
        # --- Convert to PIL for processor ---
        image_pil = Image.fromarray(image_np)
        
        # --- Map category IDs to model label space ---
        # TSR categories: 2=column, 3=row, 4=col_header, 5=proj_row_hdr, 6=spanning_cell
        # TATR model labels: 0=table, 1=column, 2=row, 3=col_header, 4=proj_row_hdr, 5=spanning_cell
        # Correct mapping: cat_id - 1  (e.g. cat 2 → model label 1, cat 3 → model label 2, ...)
        class_labels = []
        for cat_id in category_ids:
            model_label = cat_id - 1
            if 1 <= model_label <= 5:
                class_labels.append(model_label)
            else:
                class_labels.append(max(0, cat_id))  # Fallback
        
        # --- Convert bboxes to DETR format (normalized center_x, center_y, w, h) ---
        img_w, img_h = image_pil.size
        detr_boxes = []
        valid_labels = []
        
        for bbox, label in zip(bboxes, class_labels):
            x, y, w, h = bbox
            cx = (x + w / 2) / img_w
            cy = (y + h / 2) / img_h
            nw = w / img_w
            nh = h / img_h
            
            # Clamp to [0, 1]
            cx = max(0.0, min(1.0, cx))
            cy = max(0.0, min(1.0, cy))
            nw = max(0.0, min(1.0, nw))
            nh = max(0.0, min(1.0, nh))
            
            if nw > 0 and nh > 0:
                detr_boxes.append([cx, cy, nw, nh])
                valid_labels.append(label)
        
        # --- Process image through DETR processor ---
        encoding = self.processor(images=image_pil, return_tensors='pt')
        pixel_values = encoding['pixel_values'].squeeze(0)
        
        # --- Build DETR-format labels ---
        target = {
            'class_labels': torch.tensor(valid_labels, dtype=torch.long),
            'boxes': torch.tensor(detr_boxes, dtype=torch.float32) if detr_boxes else torch.zeros((0, 4), dtype=torch.float32),
        }
        
        if self.mode in ('val', 'test'):
            # Store original COCO-format boxes for evaluation
            target['orig_boxes_coco'] = bboxes
            target['orig_category_ids'] = category_ids
            target['image_id'] = entry['image_id']
            target['file_name'] = entry['file_name']
            target['orig_size'] = (img_w, img_h)
        
        return pixel_values, target
    
    def __repr__(self):
        return f"TSRTrainDataset(mode={self.mode}, crops={len(self)})"


def train_collate_fn(batch):
    """
    Collate function for DETR-style training.
    Handles variable annotation counts per image.
    """
    pixel_values_list = [item[0] for item in batch]
    targets_list = [item[1] for item in batch]
    
    # Pad pixel_values to same spatial dimensions
    max_h = max(p.shape[1] for p in pixel_values_list)
    max_w = max(p.shape[2] for p in pixel_values_list)
    
    padded_images = []
    pixel_masks = []
    
    for img in pixel_values_list:
        c, h, w = img.shape
        padded = torch.zeros((c, max_h, max_w), dtype=img.dtype)
        padded[:, :h, :w] = img
        padded_images.append(padded)
        
        mask = torch.zeros((max_h, max_w), dtype=torch.bool)
        mask[:h, :w] = True
        pixel_masks.append(mask)
    
    # Build labels list (DETR expects list of dicts)
    labels = []
    for target in targets_list:
        labels.append({
            'class_labels': target['class_labels'],
            'boxes': target['boxes'],
        })
    
    return {
        'pixel_values': torch.stack(padded_images),
        'pixel_mask': torch.stack(pixel_masks),
        'labels': labels,
        'targets': targets_list,  # Full targets for evaluation
    }


print("Data pipeline classes defined.")

In [ ]:
# =====================================================================
# Load raw COCO JSON, build crop manifest, split, create data loaders
# =====================================================================

# 1. Load COCO JSON
print(f"Loading COCO annotations from: {JSON_PATH}")
coco_data = load_coco_json(JSON_PATH)
print(f"  Images: {len(coco_data['images'])}, Annotations: {len(coco_data['annotations'])}, "
      f"Categories: {len(coco_data['categories'])}")

# 2. Filter images with too few child annotations
print(f"\nFiltering images with <= {MIN_CHILD_ANNOTATIONS} child annotations...")
coco_filtered = filter_images_by_child_count(coco_data, min_children=MIN_CHILD_ANNOTATIONS)
print(f"  After filter: {len(coco_filtered['images'])} images, "
      f"{len(coco_filtered['annotations'])} annotations")

# 3. Build crop manifest (one entry per table)
print(f"\nBuilding per-table crop manifest (padding={TABLE_PADDING}px)...")
crop_manifest = build_crop_manifest(coco_filtered)

# 4. Split at image level
print(f"\nSplitting at image level ({TRAIN_RATIO:.0%}/{VAL_RATIO:.0%}/{TEST_RATIO:.0%})...")
train_manifest, val_manifest, test_manifest = split_manifest_by_image(
    crop_manifest, TRAIN_RATIO, VAL_RATIO, TEST_RATIO, seed=SEED
)

# 5. Create DETR image processor
print(f"\nCreating DetrImageProcessor (size={IMAGE_SIZE})...")
tsr_processor = DetrImageProcessor.from_pretrained(
    TSR_MODEL_NAME,
    size={"shortest_edge": IMAGE_SIZE, "longest_edge": IMAGE_SIZE},
    do_resize=True,
    do_normalize=True,
)

# 6. Create datasets
print(f"\nCreating datasets...")
train_dataset = TSRTrainDataset(
    crop_manifest=train_manifest,
    images_dir=IMAGES_DIR,
    processor=tsr_processor,
    augmentation=train_augmentation,
    mode='train',
)

val_dataset = TSRTrainDataset(
    crop_manifest=val_manifest,
    images_dir=IMAGES_DIR,
    processor=tsr_processor,
    augmentation=None,
    mode='val',
)

test_dataset = TSRTrainDataset(
    crop_manifest=test_manifest,
    images_dir=IMAGES_DIR,
    processor=tsr_processor,
    augmentation=None,
    mode='test',
)

# 7. Create data loaders
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    collate_fn=train_collate_fn,
    drop_last=True,
    pin_memory=True,
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    collate_fn=train_collate_fn,
    pin_memory=True,
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    collate_fn=train_collate_fn,
    pin_memory=True,
)

print(f"\nDataLoaders created:")
print(f"  Train: {len(train_dataloader)} batches ({len(train_dataset)} crops)")
print(f"  Val:   {len(val_dataloader)} batches ({len(val_dataset)} crops)")
print(f"  Test:  {len(test_dataloader)} batches ({len(test_dataset)} crops)")

## 6b. Augmentation Verification

Visualize augmented samples with bounding boxes to confirm annotations survive geometric transforms.

In [ ]:
# Visualize augmented training samples to verify augmentation pipeline
print("Visualizing augmented training samples...")

num_viz_samples = min(4, len(train_dataset))
fig, axes = plt.subplots(2, num_viz_samples, figsize=(5 * num_viz_samples, 10))
if num_viz_samples == 1:
    axes = axes.reshape(2, 1)

colors = {1: 'blue', 2: 'green', 3: 'orange', 4: 'purple', 5: 'red'}
label_names = {1: 'column', 2: 'row', 3: 'col_hdr', 4: 'proj_hdr', 5: 'span_cell'}

for i in range(num_viz_samples):
    # Get sample WITHOUT augmentation (temporarily)
    original_aug = train_dataset.augmentation
    train_dataset.augmentation = None
    pv_orig, target_orig = train_dataset[i]
    train_dataset.augmentation = original_aug
    
    # Get sample WITH augmentation
    pv_aug, target_aug = train_dataset[i]
    
    # Denormalize pixel values for visualization
    for ax_idx, (pv, target, title) in enumerate([
        (pv_orig, target_orig, 'Original'),
        (pv_aug, target_aug, 'Augmented'),
    ]):
        ax = axes[ax_idx, i]
        
        # Rough denormalization for display
        img_np = pv.permute(1, 2, 0).numpy()
        img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_np = np.clip(img_np, 0, 1)
        
        ax.imshow(img_np)
        
        # Draw boxes
        h, w = img_np.shape[:2]
        for box, label in zip(target['boxes'], target['class_labels']):
            cx, cy, bw, bh = box.tolist()
            x1 = (cx - bw/2) * w
            y1 = (cy - bh/2) * h
            rect_w = bw * w
            rect_h = bh * h
            color = colors.get(label.item(), 'gray')
            rect = patches.Rectangle(
                (x1, y1), rect_w, rect_h,
                linewidth=1.5, edgecolor=color, facecolor='none'
            )
            ax.add_patch(rect)
        
        n_boxes = len(target['boxes'])
        ax.set_title(f'{title} ({n_boxes} boxes)', fontsize=10)
        ax.axis('off')

plt.suptitle('Augmentation Verification: Original vs Augmented', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("Augmentation verified — bounding boxes survive geometric transforms.")

## 7. Evaluation Metrics

Reusable metric tracking infrastructure: IoU computation, greedy prediction-to-GT matching, per-category metrics, and the `MetricsTracker` class used for both validation during training and final benchmark evaluation.

In [ ]:
def compute_iou(box1: List[float], box2: List[float]) -> float:
    """
    Compute IoU between two boxes in COCO format [x, y, w, h].
    
    Args:
        box1, box2: Boxes in [x, y, width, height] format
    
    Returns:
        IoU score (0.0 to 1.0)
    """
    # Convert to [x1, y1, x2, y2]
    x1_1, y1_1 = box1[0], box1[1]
    x2_1, y2_1 = box1[0] + box1[2], box1[1] + box1[3]
    
    x1_2, y1_2 = box2[0], box2[1]
    x2_2, y2_2 = box2[0] + box2[2], box2[1] + box2[3]
    
    # Compute intersection
    x1_i = max(x1_1, x1_2)
    y1_i = max(y1_1, y1_2)
    x2_i = min(x2_1, x2_2)
    y2_i = min(y2_1, y2_2)
    
    if x2_i <= x1_i or y2_i <= y1_i:
        return 0.0
    
    intersection = (x2_i - x1_i) * (y2_i - y1_i)
    
    # Compute union
    area1 = box1[2] * box1[3]
    area2 = box2[2] * box2[3]
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0.0


def match_predictions_to_gt(
    pred_boxes: List[List[float]],
    pred_scores: List[float],
    pred_labels: List[int],
    gt_boxes: List[List[float]],
    gt_labels: List[int],
    iou_threshold: float = 0.5
) -> Tuple[int, int, int, List[float], List[dict], List[dict]]:
    """
    Match predictions to ground truth boxes using greedy matching per category.
    
    Args:
        pred_boxes: Predicted boxes in COCO format
        pred_scores: Confidence scores for predictions
        pred_labels: Predicted category labels
        gt_boxes: Ground truth boxes in COCO format
        gt_labels: Ground truth category labels
        iou_threshold: IoU threshold for matching
    
    Returns:
        Tuple of (true_positives, false_positives, false_negatives, matched_ious, fp_details, fn_details)
    """
    if len(pred_boxes) == 0:
        fn_details = [{'gt_box': gt_boxes[i], 'gt_label': gt_labels[i]} for i in range(len(gt_boxes))]
        return 0, 0, len(gt_boxes), [], [], fn_details
    
    if len(gt_boxes) == 0:
        fp_details = [{'pred_box': pred_boxes[i], 'pred_label': pred_labels[i], 'pred_score': pred_scores[i]} 
                      for i in range(len(pred_boxes))]
        return 0, len(pred_boxes), 0, [], fp_details, []
    
    # Sort predictions by score (highest first)
    sorted_indices = np.argsort(pred_scores)[::-1]
    pred_boxes_sorted = [pred_boxes[i] for i in sorted_indices]
    pred_labels_sorted = [pred_labels[i] for i in sorted_indices]
    pred_scores_sorted = [pred_scores[i] for i in sorted_indices]
    
    matched_gt = set()
    tp = 0
    fp = 0
    matched_ious = []
    fp_details = []
    
    for pred_idx, (pred_box, pred_label, pred_score) in enumerate(zip(pred_boxes_sorted, pred_labels_sorted, pred_scores_sorted)):
        best_iou = 0
        best_gt_idx = -1
        
        for gt_idx, (gt_box, gt_label) in enumerate(zip(gt_boxes, gt_labels)):
            if gt_idx in matched_gt:
                continue
            # Only match same category (or allow matching for structure elements)
            iou = compute_iou(pred_box, gt_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx
        
        if best_iou >= iou_threshold and best_gt_idx >= 0:
            tp += 1
            matched_gt.add(best_gt_idx)
            matched_ious.append(best_iou)
        else:
            fp += 1
            fp_details.append({
                'pred_box': pred_box,
                'pred_label': pred_label,
                'pred_score': pred_score,
                'best_iou': best_iou
            })
    
    # False negatives: GT boxes not matched
    fn_details = []
    for gt_idx in range(len(gt_boxes)):
        if gt_idx not in matched_gt:
            fn_details.append({
                'gt_box': gt_boxes[gt_idx],
                'gt_label': gt_labels[gt_idx]
            })
    
    fn = len(gt_boxes) - len(matched_gt)
    
    return tp, fp, fn, matched_ious, fp_details, fn_details


def match_predictions_to_gt_per_category(
    pred_boxes: List[List[float]],
    pred_scores: List[float],
    pred_labels: List[int],
    gt_boxes: List[List[float]],
    gt_labels: List[int],
    iou_threshold: float = 0.5
) -> Dict[int, Tuple[int, int, int, List[float]]]:
    """
    Match predictions to ground truth boxes per category.
    
    Args:
        pred_boxes: Predicted boxes in COCO format
        pred_scores: Confidence scores for predictions
        pred_labels: Predicted category labels
        gt_boxes: Ground truth boxes in COCO format
        gt_labels: Ground truth category labels
        iou_threshold: IoU threshold for matching
    
    Returns:
        Dictionary mapping category_id to (tp, fp, fn, matched_ious)
    """
    # Get all unique categories
    all_categories = set(pred_labels) | set(gt_labels)
    
    category_results = {}
    
    for cat_id in all_categories:
        # Filter predictions and GT for this category
        cat_pred_indices = [i for i, label in enumerate(pred_labels) if label == cat_id]
        cat_gt_indices = [i for i, label in enumerate(gt_labels) if label == cat_id]
        
        cat_pred_boxes = [pred_boxes[i] for i in cat_pred_indices]
        cat_pred_scores = [pred_scores[i] for i in cat_pred_indices]
        cat_pred_labels = [pred_labels[i] for i in cat_pred_indices]
        
        cat_gt_boxes = [gt_boxes[i] for i in cat_gt_indices]
        cat_gt_labels = [gt_labels[i] for i in cat_gt_indices]
        
        # Match for this category
        if len(cat_pred_boxes) == 0 and len(cat_gt_boxes) == 0:
            category_results[cat_id] = (0, 0, 0, [])
            continue
        
        if len(cat_pred_boxes) == 0:
            category_results[cat_id] = (0, 0, len(cat_gt_boxes), [])
            continue
        
        if len(cat_gt_boxes) == 0:
            category_results[cat_id] = (0, len(cat_pred_boxes), 0, [])
            continue
        
        # Sort predictions by score (highest first)
        sorted_indices = np.argsort(cat_pred_scores)[::-1]
        sorted_pred_boxes = [cat_pred_boxes[i] for i in sorted_indices]
        sorted_pred_scores = [cat_pred_scores[i] for i in sorted_indices]
        
        matched_gt = set()
        tp = 0
        fp = 0
        matched_ious = []
        
        for pred_box, pred_score in zip(sorted_pred_boxes, sorted_pred_scores):
            best_iou = 0
            best_gt_idx = -1
            
            for gt_idx, gt_box in enumerate(cat_gt_boxes):
                if gt_idx in matched_gt:
                    continue
                iou = compute_iou(pred_box, gt_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx
            
            if best_iou >= iou_threshold and best_gt_idx >= 0:
                tp += 1
                matched_gt.add(best_gt_idx)
                matched_ious.append(best_iou)
            else:
                fp += 1
        
        fn = len(cat_gt_boxes) - len(matched_gt)
        category_results[cat_id] = (tp, fp, fn, matched_ious)
    
    return category_results


class MetricsTracker:
    """
    Track evaluation metrics across the dataset.
    """
    
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.total_gt = 0
        self.total_tp = 0
        self.total_fp = 0
        self.total_fn = 0
        self.all_ious = []
        self.per_image_metrics = []
        # Track per-category metrics
        self.category_metrics = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0, 'gt': 0, 'ious': []})
    
    def update(
        self,
        pred_boxes: List[List[float]],
        pred_scores: List[float],
        pred_labels: List[int],
        gt_boxes: List[List[float]],
        gt_labels: List[int],
        iou_threshold: float = 0.5
    ) -> Tuple[List[dict], List[dict]]:
        """
        Update metrics with predictions from one image.
        Returns FP and FN details for error analysis.
        """
        tp, fp, fn, matched_ious, fp_details, fn_details = match_predictions_to_gt(
            pred_boxes, pred_scores, pred_labels, gt_boxes, gt_labels, iou_threshold
        )
        
        self.total_gt += len(gt_boxes)
        self.total_tp += tp
        self.total_fp += fp
        self.total_fn += fn
        self.all_ious.extend(matched_ious)
        
        # Update per-category metrics
        category_results = match_predictions_to_gt_per_category(
            pred_boxes, pred_scores, pred_labels, gt_boxes, gt_labels, iou_threshold
        )
        
        for cat_id, (cat_tp, cat_fp, cat_fn, cat_ious) in category_results.items():
            self.category_metrics[cat_id]['tp'] += cat_tp
            self.category_metrics[cat_id]['fp'] += cat_fp
            self.category_metrics[cat_id]['fn'] += cat_fn
            self.category_metrics[cat_id]['gt'] += cat_tp + cat_fn  # GT count is TP + FN
            self.category_metrics[cat_id]['ious'].extend(cat_ious)
        
        # Store per-image metrics
        if len(gt_boxes) > 0:
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / len(gt_boxes)
            self.per_image_metrics.append({
                'tp': tp, 'fp': fp, 'fn': fn,
                'precision': precision, 'recall': recall,
                'avg_iou': np.mean(matched_ious) if matched_ious else 0
            })
        
        return fp_details, fn_details
    
    def compute_metrics(self) -> dict:
        """
        Compute final aggregated metrics.
        """
        precision = self.total_tp / (self.total_tp + self.total_fp) if (self.total_tp + self.total_fp) > 0 else 0
        recall = self.total_tp / self.total_gt if self.total_gt > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        avg_iou = np.mean(self.all_ious) if self.all_ious else 0
        
        return {
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'avg_iou': avg_iou,
            'total_tp': self.total_tp,
            'total_fp': self.total_fp,
            'total_fn': self.total_fn,
            'total_gt': self.total_gt,
        }
    
    def compute_category_metrics(self) -> Dict[int, dict]:
        """
        Compute metrics for each category individually.
        
        Returns:
            Dictionary mapping category_id to metrics dict
        """
        results = {}
        for cat_id, metrics in self.category_metrics.items():
            tp = metrics['tp']
            fp = metrics['fp']
            fn = metrics['fn']
            gt = metrics['gt']
            ious = metrics['ious']
            
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / gt if gt > 0 else 0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
            avg_iou = np.mean(ious) if ious else 0
            
            results[cat_id] = {
                'precision': precision,
                'recall': recall,
                'f1_score': f1,
                'avg_iou': avg_iou,
                'tp': tp,
                'fp': fp,
                'fn': fn,
                'gt': gt,
            }
        
        return results


print("Metrics implementation complete.")

## 8. Model Label Mapping

Map between model output labels and dataset annotation labels. The TATR model outputs labels 0-5 while our dataset uses 1-6.

In [ ]:
# The TSR model has its own label mapping that differs from our dataset
# Model labels (from microsoft/table-transformer-structure-recognition):
# 0: table, 1: table column, 2: table row, 3: table column header,
# 4: table projected row header, 5: table spanning cell

# Our dataset labels:
# 1: table, 2: table column, 3: table row, 4: table column header,
# 5: table projected row header, 6: table spanning cell

# Mapping from model label to dataset label:
MODEL_TO_DATASET_LABEL = {
    0: 1,  # table -> table
    1: 2,  # table column -> table column
    2: 3,  # table row -> table row
    3: 4,  # table column header -> table column header
    4: 5,  # table projected row header -> table projected row header
    5: 6,  # table spanning cell -> table spanning cell
}

DATASET_TO_MODEL_LABEL = {v: k for k, v in MODEL_TO_DATASET_LABEL.items()}

# Categories we evaluate (exclude table)
EVAL_MODEL_LABELS = [1, 2, 3, 4, 5]  # Model labels for structure elements
EVAL_DATASET_LABELS = [2, 3, 4, 5, 6]  # Dataset labels for structure elements

# Category names for display
structure_categories = [2, 3, 4, 5, 6]
cat_names = {
    1: "table",
    2: "table column",
    3: "table row",
    4: "table column header",
    5: "table projected row header",
    6: "table spanning cell"
}

print("Model label mapping:")
for model_label, dataset_label in MODEL_TO_DATASET_LABEL.items():
    print(f"  Model {model_label} -> Dataset {dataset_label} ({cat_names.get(dataset_label, 'unknown')})")
print(f"\nEvaluated categories: {[cat_names.get(c, f'cat_{c}') for c in structure_categories]}")

## 9. Training Loop

Core training infrastructure:
- `train_one_epoch()`: Forward pass with AMP, gradient accumulation, gradient clipping
- `evaluate_model()`: Validation loss + post-processed metric evaluation
- `run_training()`: Main loop with AdamW optimizer, cosine annealing LR, early stopping, checkpoint saving

In [ ]:
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: str,
    scaler: GradScaler,
    gradient_accumulation_steps: int = 4,
    max_grad_norm: float = 0.01,
    epoch: int = 0,
):
    """
    Train for one epoch with AMP, gradient accumulation, and gradient clipping.
    
    Returns:
        Average training loss for the epoch.
    """
    model.train()
    total_loss = 0.0
    num_batches = 0
    optimizer.zero_grad()

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1} [Train]")
    for step, batch in enumerate(pbar):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"]  # list of dicts

        # Move label tensors to device
        labels_on_device = []
        for lbl in labels:
            labels_on_device.append({
                "class_labels": lbl["class_labels"].to(device),
                "boxes": lbl["boxes"].to(device),
            })

        # Forward pass with AMP
        with autocast(enabled=(device == "cuda")):
            outputs = model(pixel_values=pixel_values, labels=labels_on_device)
            loss = outputs.loss / gradient_accumulation_steps

        # Backward pass
        scaler.scale(loss).backward()

        # Gradient accumulation step
        if (step + 1) % gradient_accumulation_steps == 0 or (step + 1) == len(dataloader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * gradient_accumulation_steps
        num_batches += 1
        pbar.set_postfix({"loss": f"{total_loss / num_batches:.4f}"})

    avg_loss = total_loss / max(num_batches, 1)
    return avg_loss


@torch.no_grad()
def evaluate_model(
    model: nn.Module,
    dataloader: DataLoader,
    device: str,
    processor: DetrImageProcessor,
    confidence_threshold: float = 0.5,
    iou_threshold: float = 0.5,
):
    """
    Evaluate model on validation set. Computes loss + post-processed metrics.
    
    Returns:
        Dictionary with val_loss, precision, recall, f1, avg_iou.
    """
    model.eval()
    total_loss = 0.0
    num_batches = 0
    metrics_tracker = MetricsTracker()

    for batch in tqdm(dataloader, desc="Evaluating"):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"]

        labels_on_device = []
        for lbl in labels:
            labels_on_device.append({
                "class_labels": lbl["class_labels"].to(device),
                "boxes": lbl["boxes"].to(device),
            })

        # Compute loss
        outputs = model(pixel_values=pixel_values, labels=labels_on_device)
        total_loss += outputs.loss.item()
        num_batches += 1

        # Post-process for metrics
        target_sizes = torch.tensor(
            [[IMAGE_SIZE, IMAGE_SIZE]] * pixel_values.shape[0],
            device=device,
            dtype=torch.float32,
        )

        results_batch = processor.post_process_object_detection(
            outputs,
            threshold=confidence_threshold,
            target_sizes=target_sizes,
        )

        for i, results in enumerate(results_batch):
            pred_boxes_xyxy = results["boxes"].cpu().numpy()
            pred_scores = results["scores"].cpu().numpy()
            pred_labels_model = results["labels"].cpu().numpy()

            # Convert predictions to COCO format [x, y, w, h] in dataset label space
            pred_boxes_coco = []
            pred_scores_list = []
            pred_labels_dataset = []

            for box, score, label in zip(pred_boxes_xyxy, pred_scores, pred_labels_model):
                if int(label) == 0:  # skip 'table' category
                    continue
                x1, y1, x2, y2 = box
                pred_boxes_coco.append([float(x1), float(y1), float(x2 - x1), float(y2 - y1)])
                pred_scores_list.append(float(score))
                pred_labels_dataset.append(MODEL_TO_DATASET_LABEL.get(int(label), int(label)))

            # Get GT from batch labels
            gt_boxes_norm = labels[i]["boxes"].cpu().numpy()
            gt_labels_raw = labels[i]["class_labels"].cpu().numpy()

            gt_boxes_coco = []
            gt_labels_dataset = []
            for gt_box, gt_label in zip(gt_boxes_norm, gt_labels_raw):
                cx, cy, bw, bh = gt_box
                x = (cx - bw / 2) * IMAGE_SIZE
                y = (cy - bh / 2) * IMAGE_SIZE
                w = bw * IMAGE_SIZE
                h = bh * IMAGE_SIZE
                gt_boxes_coco.append([x, y, w, h])
                gt_labels_dataset.append(MODEL_TO_DATASET_LABEL.get(int(gt_label), int(gt_label)))

            # Update metrics
            metrics_tracker.update(
                pred_boxes_coco, pred_scores_list, pred_labels_dataset,
                gt_boxes_coco, gt_labels_dataset, iou_threshold=iou_threshold,
            )

    avg_loss = total_loss / max(num_batches, 1)
    metrics = metrics_tracker.compute_metrics()
    metrics["val_loss"] = avg_loss

    return metrics


def run_training(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    processor: DetrImageProcessor,
    device: str,
    max_epochs: int = 500,
    learning_rate: float = 1e-3,
    weight_decay: float = 1e-5,
    gradient_accumulation_steps: int = 4,
    max_grad_norm: float = 0.01,
    early_stopping_patience: int = 10,
    confidence_threshold: float = 0.5,
    checkpoint_dir: str = None,
):
    """
    Main training loop with AdamW, cosine annealing, early stopping, checkpointing.
    
    Returns:
        training_history dict with per-epoch metrics and best checkpoint path.
    """
    model.to(device)

    # Optimizer: only trainable parameters (LoRA adapters)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    total_params = sum(p.numel() for p in model.parameters())
    trainable_count = sum(p.numel() for p in trainable_params)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_count:,} ({100 * trainable_count / total_params:.2f}%)")

    optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs, eta_min=1e-6)
    scaler = GradScaler(enabled=(device == "cuda"))

    # Checkpoint directory
    if checkpoint_dir is None:
        checkpoint_dir = os.path.join(OUTPUT_DIR, "checkpoints")
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Training history
    history = {
        "train_loss": [],
        "val_loss": [],
        "val_precision": [],
        "val_recall": [],
        "val_f1": [],
        "val_avg_iou": [],
        "learning_rate": [],
        "best_epoch": 0,
        "best_val_f1": 0.0,
        "best_checkpoint": "",
    }

    best_val_f1 = 0.0
    patience_counter = 0

    print(f"\n{'='*70}")
    print(f"STARTING TRAINING — TATR + LoRA")
    print(f"{'='*70}")
    print(f"Max epochs: {max_epochs}")
    print(f"Learning rate: {learning_rate}")
    print(f"Gradient accumulation: {gradient_accumulation_steps}")
    print(f"Max grad norm: {max_grad_norm}")
    print(f"Early stopping patience: {early_stopping_patience}")
    print(f"Device: {device}")
    print(f"{'='*70}\n")

    for epoch in range(max_epochs):
        # Train
        train_loss = train_one_epoch(
            model, train_loader, optimizer, device, scaler,
            gradient_accumulation_steps=gradient_accumulation_steps,
            max_grad_norm=max_grad_norm,
            epoch=epoch,
        )

        # Validate
        val_metrics = evaluate_model(
            model, val_loader, device, processor,
            confidence_threshold=confidence_threshold,
        )

        # Update scheduler
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        # Record history
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["val_loss"])
        history["val_precision"].append(val_metrics["precision"])
        history["val_recall"].append(val_metrics["recall"])
        history["val_f1"].append(val_metrics["f1_score"])
        history["val_avg_iou"].append(val_metrics["avg_iou"])
        history["learning_rate"].append(current_lr)

        # Print epoch summary
        print(
            f"Epoch {epoch+1}/{max_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_metrics['val_loss']:.4f} | "
            f"Precision: {val_metrics['precision']:.4f} | "
            f"Recall: {val_metrics['recall']:.4f} | "
            f"F1: {val_metrics['f1_score']:.4f} | "
            f"IoU: {val_metrics['avg_iou']:.4f} | "
            f"LR: {current_lr:.2e}"
        )

        # Early stopping & checkpointing on val F1
        if val_metrics["f1_score"] > best_val_f1:
            best_val_f1 = val_metrics["f1_score"]
            patience_counter = 0
            history["best_epoch"] = epoch + 1
            history["best_val_f1"] = best_val_f1

            # Save best checkpoint
            ckpt_path = os.path.join(checkpoint_dir, "best_model.pt")
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "val_f1": best_val_f1,
                "val_metrics": val_metrics,
                "history": history,
            }, ckpt_path)
            history["best_checkpoint"] = ckpt_path
            print(f"  -> New best F1! Saved checkpoint to {ckpt_path}")
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print(f"\nEarly stopping at epoch {epoch+1} (patience={early_stopping_patience})")
                break

        # Save periodic checkpoint every 25 epochs
        if (epoch + 1) % 25 == 0:
            periodic_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch+1}.pt")
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "history": history,
            }, periodic_path)
            print(f"  -> Saved periodic checkpoint: {periodic_path}")

    print(f"\n{'='*70}")
    print(f"TRAINING COMPLETE")
    print(f"Best epoch: {history['best_epoch']} with Val F1: {history['best_val_f1']:.4f}")
    print(f"Best checkpoint: {history['best_checkpoint']}")
    print(f"{'='*70}")

    return history

## 9b. Build Model, Apply LoRA, and Train

Load the vanilla TATR model, apply LoRA via PEFT, then launch training.

In [ ]:
# =========================================================
# Step 1: Load vanilla TATR model
# =========================================================
print(f"Loading base model: {TSR_MODEL_NAME}")
base_model = TableTransformerForObjectDetection.from_pretrained(TSR_MODEL_NAME)
base_model.to(DEVICE)

# =========================================================
# Step 2: Apply LoRA via PEFT
# =========================================================
print("\nValidating LoRA target modules...")
target_modules = get_lora_target_modules(base_model)

print("\nApplying LoRA to TATR model...")
lora_model = apply_lora_to_model(
    model=base_model,
    target_modules=target_modules,
)

# Print trainable parameters summary
total_params = sum(p.numel() for p in lora_model.parameters())
trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params
print(f"\nParameter Summary:")
print(f"  Total parameters:     {total_params:>12,}")
print(f"  Trainable (LoRA):     {trainable_params:>12,} ({100*trainable_params/total_params:.2f}%)")
print(f"  Frozen:               {frozen_params:>12,} ({100*frozen_params/total_params:.2f}%)")

# =========================================================
# Step 3: Run Training
# =========================================================
print("\nLaunching training...")
training_history = run_training(
    model=lora_model,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    processor=tsr_processor,
    device=DEVICE,
    max_epochs=MAX_EPOCHS,
    learning_rate=LEARNING_RATE_LORA,
    weight_decay=WEIGHT_DECAY,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    max_grad_norm=MAX_GRAD_NORM,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    checkpoint_dir=CHECKPOINT_DIR,
)

## 10. Training Curves

Visualize training loss, validation loss, validation F1-score, and learning rate schedule across epochs.

In [ ]:
# Plot training curves
epochs_range = range(1, len(training_history["train_loss"]) + 1)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Training & Validation Loss
axes[0, 0].plot(epochs_range, training_history["train_loss"], label="Train Loss", color="steelblue", linewidth=2)
axes[0, 0].plot(epochs_range, training_history["val_loss"], label="Val Loss", color="coral", linewidth=2)
best_epoch = training_history["best_epoch"]
axes[0, 0].axvline(x=best_epoch, color="green", linestyle="--", alpha=0.7, label=f"Best epoch ({best_epoch})")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].set_title("Training & Validation Loss")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Validation F1-Score
axes[0, 1].plot(epochs_range, training_history["val_f1"], label="Val F1", color="green", linewidth=2)
axes[0, 1].axvline(x=best_epoch, color="green", linestyle="--", alpha=0.7)
axes[0, 1].axhline(y=training_history["best_val_f1"], color="red", linestyle=":", alpha=0.5, 
                     label=f"Best F1: {training_history['best_val_f1']:.4f}")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("F1-Score")
axes[0, 1].set_title("Validation F1-Score")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Precision & Recall
axes[1, 0].plot(epochs_range, training_history["val_precision"], label="Val Precision", color="blue", linewidth=2)
axes[1, 0].plot(epochs_range, training_history["val_recall"], label="Val Recall", color="orange", linewidth=2)
axes[1, 0].axvline(x=best_epoch, color="green", linestyle="--", alpha=0.7, label=f"Best epoch ({best_epoch})")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Score")
axes[1, 0].set_title("Validation Precision & Recall")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Learning Rate Schedule
axes[1, 1].plot(epochs_range, training_history["learning_rate"], color="purple", linewidth=2)
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Learning Rate")
axes[1, 1].set_title("Cosine Annealing LR Schedule")
axes[1, 1].set_yscale("log")
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle("TATR + LoRA Training Curves", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"Training curves saved to: {os.path.join(OUTPUT_DIR, 'training_curves.png')}")
print(f"Best epoch: {best_epoch}, Best Val F1: {training_history['best_val_f1']:.4f}")

In [ ]:
# ================================================================
# THREE-WAY BENCHMARK EVALUATION INFRASTRUCTURE
# ================================================================

def run_benchmark_inference(
    model: nn.Module,
    dataloader: DataLoader,
    processor_bm: DetrImageProcessor,
    device: str,
    confidence_threshold: float = 0.5,
    apply_postprocessing: bool = True,
    model_name: str = "model",
):
    """
    Run inference for benchmark evaluation. Returns metrics at IoU=0.50/0.75 and mAP@50-95.
    """
    model.eval()
    model.to(device)

    metrics_50 = MetricsTracker()
    metrics_75 = MetricsTracker()
    iou_thresholds = np.arange(0.5, 1.0, 0.05)
    metrics_per_threshold = {t: MetricsTracker() for t in iou_thresholds}
    
    predictions = []
    error_samples = []
    postprocessing_stats = {
        'total_removed_by_confidence': 0,
        'total_removed_by_nms': 0,
        'total_removed_spanning': 0,
        'total_removed_headers': 0,
    }
    
    initial_threshold = 0.3 if apply_postprocessing else confidence_threshold

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Benchmark [{model_name}]"):
            pixel_values = batch["pixel_values"].to(device)
            
            # Handle both training-style batches (with labels) and inference-style
            if "labels" in batch:
                outputs = model(pixel_values=pixel_values)
            else:
                pixel_mask = batch.get("pixel_mask")
                if pixel_mask is not None:
                    pixel_mask = pixel_mask.to(device)
                outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask)

            batch_size = pixel_values.shape[0]
            target_sizes = torch.tensor(
                [[IMAGE_SIZE, IMAGE_SIZE]] * batch_size,
                device=device, dtype=torch.float32,
            )

            results_batch = processor_bm.post_process_object_detection(
                outputs, threshold=initial_threshold, target_sizes=target_sizes,
            )

            for i in range(batch_size):
                results = results_batch[i]
                pred_boxes_xyxy = results["boxes"].cpu().numpy()
                pred_scores_np = results["scores"].cpu().numpy()
                pred_labels_model = results["labels"].cpu().numpy()

                # Convert to COCO format + dataset labels
                pred_boxes_coco = []
                pred_scores_list = []
                pred_labels_dataset = []
                for box, score, label in zip(pred_boxes_xyxy, pred_scores_np, pred_labels_model):
                    if int(label) == 0:
                        continue
                    x1, y1, x2, y2 = box
                    pred_boxes_coco.append([float(x1), float(y1), float(x2 - x1), float(y2 - y1)])
                    pred_scores_list.append(float(score))
                    pred_labels_dataset.append(MODEL_TO_DATASET_LABEL.get(int(label), int(label)))

                # Get GT boxes
                if "labels" in batch:
                    gt_boxes_norm = batch["labels"][i]["boxes"].cpu().numpy()
                    gt_labels_raw = batch["labels"][i]["class_labels"].cpu().numpy()
                    gt_boxes_coco = []
                    gt_labels_dataset = []
                    for gt_box, gt_label in zip(gt_boxes_norm, gt_labels_raw):
                        cx, cy, bw, bh = gt_box
                        x = (cx - bw / 2) * IMAGE_SIZE
                        y = (cy - bh / 2) * IMAGE_SIZE
                        w = bw * IMAGE_SIZE
                        h = bh * IMAGE_SIZE
                        gt_boxes_coco.append([x, y, w, h])
                        gt_labels_dataset.append(MODEL_TO_DATASET_LABEL.get(int(gt_label), int(gt_label)))
                else:
                    gt_boxes_coco = batch["gt_boxes"][i]
                    gt_labels_dataset = batch["gt_labels"][i]

                # Apply post-processing
                if apply_postprocessing and len(pred_boxes_coco) > 0:
                    (pred_boxes_coco, pred_scores_list, pred_labels_dataset, diagnostics
                    ) = apply_table_structure_postprocessing(
                        pred_boxes_coco, pred_scores_list, pred_labels_dataset,
                        image_size=(IMAGE_SIZE, IMAGE_SIZE),
                    )
                    postprocessing_stats['total_removed_by_confidence'] += diagnostics['removed_by_confidence']
                    postprocessing_stats['total_removed_by_nms'] += diagnostics['removed_by_nms']
                    postprocessing_stats['total_removed_spanning'] += diagnostics['removed_spanning_cells']
                    postprocessing_stats['total_removed_headers'] += diagnostics['removed_headers']

                # Update metrics
                fp_details_50, fn_details_50 = metrics_50.update(
                    pred_boxes_coco, pred_scores_list, pred_labels_dataset,
                    gt_boxes_coco, gt_labels_dataset, iou_threshold=0.5,
                )
                metrics_75.update(
                    pred_boxes_coco, pred_scores_list, pred_labels_dataset,
                    gt_boxes_coco, gt_labels_dataset, iou_threshold=0.75,
                )
                for t in iou_thresholds:
                    metrics_per_threshold[t].update(
                        pred_boxes_coco, pred_scores_list, pred_labels_dataset,
                        gt_boxes_coco, gt_labels_dataset, iou_threshold=t,
                    )

                predictions.append({
                    "pred_boxes": pred_boxes_coco,
                    "pred_scores": pred_scores_list,
                    "pred_labels": pred_labels_dataset,
                    "gt_boxes": gt_boxes_coco,
                    "gt_labels": gt_labels_dataset,
                })
                
                if len(fp_details_50) > 0 or len(fn_details_50) > 0:
                    error_samples.append({
                        "pred_boxes": pred_boxes_coco,
                        "pred_scores": pred_scores_list,
                        "pred_labels": pred_labels_dataset,
                        "gt_boxes": gt_boxes_coco,
                        "gt_labels": gt_labels_dataset,
                        "fp_details": fp_details_50,
                        "fn_details": fn_details_50,
                    })

    # Compute mAP@50-95
    aps = []
    for t in iou_thresholds:
        m = metrics_per_threshold[t].compute_metrics()
        aps.append(m["precision"] * m["recall"])
    mAP_50_95 = float(np.mean(aps)) if aps else 0.0

    results_50 = metrics_50.compute_metrics()
    results_75 = metrics_75.compute_metrics()
    category_results_50 = metrics_50.compute_category_metrics()

    return {
        "model_name": model_name,
        "results_50": results_50,
        "results_75": results_75,
        "category_results_50": category_results_50,
        "mAP_50_95": mAP_50_95,
        "predictions": predictions,
        "error_samples": error_samples,
        "postprocessing_stats": postprocessing_stats,
    }


print("Benchmark inference function defined.")

In [ ]:
# ================================================================
# TWO-WAY BENCHMARK EXECUTION
# ================================================================
print("=" * 70)
print("BENCHMARK EVALUATION ON VALIDATION SET")
print("=" * 70)

benchmark_results = {}

# ---- Baseline A: Vanilla TATR (no LoRA) ----
print("\n[A] Loading vanilla TATR baseline...")
vanilla_model = TableTransformerForObjectDetection.from_pretrained(TSR_MODEL_NAME)
vanilla_model.to(DEVICE)
benchmark_results["A_Vanilla_TATR"] = run_benchmark_inference(
    model=vanilla_model,
    dataloader=val_dataloader,
    processor_bm=tsr_processor,
    device=DEVICE,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    apply_postprocessing=True,
    model_name="Vanilla TATR",
)
del vanilla_model
torch.cuda.empty_cache() if DEVICE == "cuda" else None
print(f"  F1@50: {benchmark_results['A_Vanilla_TATR']['results_50']['f1_score']:.4f}")

# ---- Model B: TATR + LoRA (trained) ----
print("\n[B] Loading best trained TATR + LoRA checkpoint...")
best_ckpt_path = training_history["best_checkpoint"]
if best_ckpt_path and os.path.exists(best_ckpt_path):
    # Reload model and apply LoRA, then load weights
    trained_base = TableTransformerForObjectDetection.from_pretrained(TSR_MODEL_NAME)
    trained_base.to(DEVICE)
    target_modules = get_lora_target_modules(trained_base)
    trained_model = apply_lora_to_model(
        trained_base,
        target_modules=target_modules,
    )
    checkpoint = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
    trained_model.load_state_dict(checkpoint["model_state_dict"])
    trained_model.to(DEVICE)

    benchmark_results["B_LoRA_Trained"] = run_benchmark_inference(
        model=trained_model,
        dataloader=val_dataloader,
        processor_bm=tsr_processor,
        device=DEVICE,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        apply_postprocessing=True,
        model_name="TATR + LoRA (trained)",
    )
    del trained_model
    torch.cuda.empty_cache() if DEVICE == "cuda" else None
    print(f"  F1@50: {benchmark_results['B_LoRA_Trained']['results_50']['f1_score']:.4f}")
else:
    print("  WARNING: No best checkpoint found. Skipping trained model evaluation.")

print("\nBenchmark complete!")

## 11. Benchmark Comparison

Side-by-side comparison of Vanilla TATR and TATR + LoRA (trained) across all metrics and categories.

In [ ]:
# ================================================================
# BENCHMARK COMPARISON TABLE
# ================================================================
print("=" * 90)
print("BENCHMARK COMPARISON")
print("=" * 90)

comparison_rows = []
for key, label in [
    ("A_Vanilla_TATR", "Vanilla TATR"),
    ("B_LoRA_Trained", "TATR + LoRA"),
]:
    if key not in benchmark_results:
        continue
    r50 = benchmark_results[key]["results_50"]
    r75 = benchmark_results[key]["results_75"]
    mAP = benchmark_results[key]["mAP_50_95"]
    comparison_rows.append({
        "Model": label,
        "P@50": r50["precision"],
        "R@50": r50["recall"],
        "F1@50": r50["f1_score"],
        "AvgIoU": r50["avg_iou"],
        "P@75": r75["precision"],
        "R@75": r75["recall"],
        "F1@75": r75["f1_score"],
        "mAP@50-95": mAP,
    })

comparison_df = pd.DataFrame(comparison_rows)
# Format numeric columns
for col in comparison_df.columns:
    if col != "Model":
        comparison_df[col] = comparison_df[col].apply(lambda x: f"{x:.4f}")
print(comparison_df.to_string(index=False))

# Per-category breakdown
print("\n" + "=" * 90)
print("PER-CATEGORY F1-SCORE BREAKDOWN (IoU=0.50)")
print("=" * 90)

cat_comparison_rows = []
for cat_id in structure_categories:
    cat_name = cat_names.get(cat_id, f"cat_{cat_id}")
    row = {"Category": cat_name}
    for key, label in [
        ("A_Vanilla_TATR", "Vanilla"),
        ("B_LoRA_Trained", "LoRA"),
    ]:
        if key not in benchmark_results:
            continue
        cat_results = benchmark_results[key].get("category_results_50", {})
        if cat_id in cat_results:
            row[f"{label} F1"] = f"{cat_results[cat_id]['f1_score']:.4f}"
            row[f"{label} P"] = f"{cat_results[cat_id]['precision']:.4f}"
            row[f"{label} R"] = f"{cat_results[cat_id]['recall']:.4f}"
        else:
            row[f"{label} F1"] = "N/A"
            row[f"{label} P"] = "N/A"
            row[f"{label} R"] = "N/A"
    cat_comparison_rows.append(row)

cat_comparison_df = pd.DataFrame(cat_comparison_rows)
print(cat_comparison_df.to_string(index=False))

# ================================================================
# BAR CHART COMPARISON
# ================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

model_labels = []
f1_50_vals = []
f1_75_vals = []
mAP_vals = []

for key, label in [
    ("A_Vanilla_TATR", "Vanilla\nTATR"),
    ("B_LoRA_Trained", "TATR\n+ LoRA"),
]:
    if key not in benchmark_results:
        continue
    model_labels.append(label)
    r50 = benchmark_results[key]["results_50"]
    r75 = benchmark_results[key]["results_75"]
    f1_50_vals.append(r50["f1_score"])
    f1_75_vals.append(r75["f1_score"])
    mAP_vals.append(benchmark_results[key]["mAP_50_95"])

x = np.arange(len(model_labels))
bar_colors = ["#4C72B0", "#55A868"][:len(model_labels)]

# F1@50
bars = axes[0].bar(x, f1_50_vals, color=bar_colors, edgecolor="navy", linewidth=1.2)
for bar, val in zip(bars, f1_50_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{val:.4f}",
                 ha="center", va="bottom", fontsize=11, fontweight="bold")
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_labels, fontsize=10)
axes[0].set_ylim(0, 1.15)
axes[0].set_ylabel("F1-Score", fontsize=12)
axes[0].set_title("F1-Score @ IoU=0.50", fontsize=14)
axes[0].grid(axis="y", alpha=0.3)

# F1@75
bars = axes[1].bar(x, f1_75_vals, color=bar_colors, edgecolor="navy", linewidth=1.2)
for bar, val in zip(bars, f1_75_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{val:.4f}",
                 ha="center", va="bottom", fontsize=11, fontweight="bold")
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_labels, fontsize=10)
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel("F1-Score", fontsize=12)
axes[1].set_title("F1-Score @ IoU=0.75", fontsize=14)
axes[1].grid(axis="y", alpha=0.3)

# mAP@50-95
bars = axes[2].bar(x, mAP_vals, color=bar_colors, edgecolor="navy", linewidth=1.2)
for bar, val in zip(bars, mAP_vals):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{val:.4f}",
                 ha="center", va="bottom", fontsize=11, fontweight="bold")
axes[2].set_xticks(x)
axes[2].set_xticklabels(model_labels, fontsize=10)
axes[2].set_ylim(0, 1.15)
axes[2].set_ylabel("mAP", fontsize=12)
axes[2].set_title("mAP @ 50-95", fontsize=14)
axes[2].grid(axis="y", alpha=0.3)

plt.suptitle("Benchmark Comparison: Vanilla vs LoRA", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "benchmark_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

# Per-category F1 grouped bar chart
fig, ax = plt.subplots(figsize=(14, 6))
short_cat_names = ["Column", "Row", "Col Hdr", "Proj Hdr", "Span Cell"]
x_cats = np.arange(len(structure_categories))
n_models = len(model_labels)
bar_width = 0.35

for m_idx, (key, m_label) in enumerate([
    ("A_Vanilla_TATR", "Vanilla"),
    ("B_LoRA_Trained", "LoRA"),
]):
    if key not in benchmark_results:
        continue
    cat_f1s = []
    for cat_id in structure_categories:
        cat_res = benchmark_results[key].get("category_results_50", {})
        cat_f1s.append(cat_res[cat_id]["f1_score"] if cat_id in cat_res else 0.0)
    
    offset = (m_idx - n_models / 2 + 0.5) * bar_width
    bars = ax.bar(x_cats + offset, cat_f1s, bar_width, label=m_label, color=bar_colors[m_idx], edgecolor="navy")
    for bar, val in zip(bars, cat_f1s):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{val:.2f}",
                    ha="center", va="bottom", fontsize=9)

ax.set_xticks(x_cats)
ax.set_xticklabels(short_cat_names, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel("F1-Score @ IoU=0.50", fontsize=12)
ax.set_title("Per-Category F1-Score Comparison", fontsize=14)
ax.legend(loc="upper right", fontsize=11)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "per_category_benchmark.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"\nBenchmark visualizations saved!")

## 12. IoU Distribution & Performance Summary (Best Model)

In [ ]:
# Use the best model's benchmark results for detailed visualization
best_key = "B_LoRA_Trained" if "B_LoRA_Trained" in benchmark_results else list(benchmark_results.keys())[-1]
best_bm = benchmark_results[best_key]
best_r50 = best_bm["results_50"]
best_r75 = best_bm["results_75"]
best_mAP = best_bm["mAP_50_95"]

print(f"Detailed metrics for: {best_bm['model_name']}")

# Collect IoU values from the benchmark run's predictions
# (We need to recompute matched_ious from predictions)
all_matched_ious = []
for pred in best_bm["predictions"]:
    _, _, _, matched_ious, _, _ = match_predictions_to_gt(
        pred["pred_boxes"], pred["pred_scores"], pred["pred_labels"],
        pred["gt_boxes"], pred["gt_labels"], 0.5
    )
    all_matched_ious.extend(matched_ious)

if all_matched_ious:
    all_ious_arr = np.array(all_matched_ious)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # IoU histogram
    axes[0].hist(all_ious_arr, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[0].axvline(x=np.mean(all_ious_arr), color='red', linestyle='--',
                    label=f'Mean: {np.mean(all_ious_arr):.3f}')
    axes[0].axvline(x=0.5, color='green', linestyle='--', label='IoU=0.5')
    axes[0].axvline(x=0.75, color='orange', linestyle='--', label='IoU=0.75')
    axes[0].set_xlabel('IoU Score', fontsize=12)
    axes[0].set_ylabel('Frequency', fontsize=12)
    axes[0].set_title(f'IoU Distribution — {best_bm["model_name"]}', fontsize=13)
    axes[0].legend()
    
    # Summary bar chart
    metrics_names = ['P@50', 'R@50', 'F1@50', 'AvgIoU', 'mAP@50', 'mAP@75', 'mAP@50-95']
    metrics_values = [
        best_r50['precision'], best_r50['recall'], best_r50['f1_score'],
        best_r50['avg_iou'],
        best_r50['precision'] * best_r50['recall'],
        best_r75['precision'] * best_r75['recall'],
        best_mAP,
    ]
    
    bar_colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(metrics_names)))
    bars = axes[1].bar(metrics_names, metrics_values, color=bar_colors, edgecolor='navy', linewidth=1.2)
    for bar, val in zip(bars, metrics_values):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.3f}',
                     ha='center', va='bottom', fontsize=10, fontweight='bold')
    axes[1].set_ylim(0, 1.15)
    axes[1].set_ylabel('Score', fontsize=12)
    axes[1].set_title(f'Performance — {best_bm["model_name"]}', fontsize=13)
    axes[1].tick_params(axis='x', rotation=20)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'best_model_performance.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No matched IoUs to visualize.")

## 13. Error Analysis (Best Model)

Analyze false positives and false negatives from the best model's benchmark predictions.

In [ ]:
# Error analysis on best model benchmark results
best_error_samples = best_bm.get("error_samples", [])
best_predictions = best_bm.get("predictions", [])

print(f"Error Analysis for: {best_bm['model_name']}")
print(f"Total test samples: {len(best_predictions)}")
print(f"Samples with errors (FP or FN): {len(best_error_samples)}")

if best_error_samples:
    total_fp = sum(len(e.get("fp_details", [])) for e in best_error_samples)
    total_fn = sum(len(e.get("fn_details", [])) for e in best_error_samples)
    print(f"Total false positives: {total_fp}")
    print(f"Total false negatives: {total_fn}")
    
    # Per-category error breakdown
    fp_by_cat = defaultdict(int)
    fn_by_cat = defaultdict(int)
    for e in best_error_samples:
        for fp in e.get("fp_details", []):
            fp_by_cat[cat_names.get(fp.get("pred_label", -1), "unknown")] += 1
        for fn in e.get("fn_details", []):
            fn_by_cat[cat_names.get(fn.get("gt_label", -1), "unknown")] += 1
    
    print("\nFP breakdown by category:")
    for cat, count in sorted(fp_by_cat.items(), key=lambda x: -x[1]):
        print(f"  {cat}: {count}")
    
    print("\nFN breakdown by category:")
    for cat, count in sorted(fn_by_cat.items(), key=lambda x: -x[1]):
        print(f"  {cat}: {count}")
    
    # Error severity histogram
    error_counts = [len(e.get("fp_details", [])) + len(e.get("fn_details", [])) for e in best_error_samples]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(error_counts, bins=20, color="coral", edgecolor="white", alpha=0.8)
    axes[0].set_xlabel("Total Errors (FP+FN) per Sample")
    axes[0].set_ylabel("Frequency")
    axes[0].set_title("Error Severity Distribution")
    axes[0].grid(axis="y", alpha=0.3)
    
    # FP vs FN by category
    all_cats = sorted(set(list(fp_by_cat.keys()) + list(fn_by_cat.keys())))
    x_pos = np.arange(len(all_cats))
    fp_vals = [fp_by_cat.get(c, 0) for c in all_cats]
    fn_vals = [fn_by_cat.get(c, 0) for c in all_cats]
    
    w = 0.35
    axes[1].bar(x_pos - w/2, fp_vals, w, label="FP", color="red", alpha=0.7)
    axes[1].bar(x_pos + w/2, fn_vals, w, label="FN", color="blue", alpha=0.7)
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels([c[:12] for c in all_cats], rotation=30, ha="right")
    axes[1].set_ylabel("Count")
    axes[1].set_title("FP vs FN by Category")
    axes[1].legend()
    axes[1].grid(axis="y", alpha=0.3)
    
    plt.suptitle(f"Error Analysis — {best_bm['model_name']}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "error_analysis.png"), dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No errors found — perfect performance on test set!")

## 14. Save Results & Error Details

In [ ]:
# Save comprehensive benchmark results JSON
benchmark_summary = {}
for key, bm in benchmark_results.items():
    benchmark_summary[key] = {
        "model_name": bm["model_name"],
        "results_50": bm["results_50"],
        "results_75": bm["results_75"],
        "mAP_50_95": bm["mAP_50_95"],
        "postprocessing_stats": bm["postprocessing_stats"],
        "num_predictions": len(bm["predictions"]),
        "num_error_samples": len(bm["error_samples"]),
    }

# Add training metadata
results_dict = {
    "experiment": "TATR + LoRA Fine-tuning for TSR",
    "base_model": TSR_MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "lora_config": {
        "r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT,
    },
    "training_config": {
        "max_epochs": MAX_EPOCHS,
        "learning_rate": LEARNING_RATE_LORA,
        "weight_decay": WEIGHT_DECAY,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "max_grad_norm": MAX_GRAD_NORM,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "batch_size": BATCH_SIZE,
    },
    "training_history": {
        "best_epoch": training_history["best_epoch"],
        "best_val_f1": training_history["best_val_f1"],
        "total_epochs": len(training_history["train_loss"]),
    },
    "dataset": {
        "train_size": len(train_dataset),
        "val_size": len(val_dataset),
        "test_size": len(test_dataset),
    },
    "benchmark_results": benchmark_summary,
}

results_path = os.path.join(OUTPUT_DIR, "lora_benchmark_results.json")
with open(results_path, "w") as f:
    json.dump(results_dict, f, indent=2, default=str)
print(f"Benchmark results saved to: {results_path}")

# Save training history for reproducibility
history_path = os.path.join(OUTPUT_DIR, "training_history.json")
with open(history_path, "w") as f:
    json.dump(training_history, f, indent=2, default=str)
print(f"Training history saved to: {history_path}")

In [ ]:
# Save error details CSV for best model
error_records = []

for idx, error_sample in enumerate(best_error_samples):
    base_record = {
        "sample_idx": idx,
        "total_gt": len(error_sample["gt_boxes"]),
        "total_preds": len(error_sample["pred_boxes"]),
    }
    
    for j, fp in enumerate(error_sample.get("fp_details", [])):
        fp_record = base_record.copy()
        fp_record["error_type"] = "FP"
        fp_record["error_idx"] = j
        fp_record["category_id"] = fp.get("pred_label", -1)
        fp_record["category_name"] = cat_names.get(fp.get("pred_label", -1), "unknown")
        fp_record["bbox_x"] = fp["pred_box"][0] if "pred_box" in fp else 0
        fp_record["bbox_y"] = fp["pred_box"][1] if "pred_box" in fp else 0
        fp_record["bbox_w"] = fp["pred_box"][2] if "pred_box" in fp else 0
        fp_record["bbox_h"] = fp["pred_box"][3] if "pred_box" in fp else 0
        fp_record["confidence"] = fp.get("pred_score", 0)
        fp_record["best_iou"] = fp.get("best_iou", 0)
        error_records.append(fp_record)
    
    for j, fn in enumerate(error_sample.get("fn_details", [])):
        fn_record = base_record.copy()
        fn_record["error_type"] = "FN"
        fn_record["error_idx"] = j
        fn_record["category_id"] = fn.get("gt_label", -1)
        fn_record["category_name"] = cat_names.get(fn.get("gt_label", -1), "unknown")
        fn_record["bbox_x"] = fn["gt_box"][0] if "gt_box" in fn else 0
        fn_record["bbox_y"] = fn["gt_box"][1] if "gt_box" in fn else 0
        fn_record["bbox_w"] = fn["gt_box"][2] if "gt_box" in fn else 0
        fn_record["bbox_h"] = fn["gt_box"][3] if "gt_box" in fn else 0
        fn_record["confidence"] = None
        fn_record["best_iou"] = None
        error_records.append(fn_record)

if error_records:
    error_df = pd.DataFrame(error_records)
    csv_path = os.path.join(OUTPUT_DIR, "tsr_lora_error_details.csv")
    error_df.to_csv(csv_path, index=False)
    
    print(f"Error details saved to: {csv_path}")
    print(f"Total error records: {len(error_records)}")
    print(f"\nError summary by type:")
    print(error_df["error_type"].value_counts())
    print(f"\nError summary by category:")
    print(error_df.groupby(["error_type", "category_name"]).size().unstack(fill_value=0))
else:
    print("No errors to save — perfect performance!")

## 15. Final Summary

Comprehensive summary of the TATR + LoRA fine-tuning experiment including architecture details, training configuration, and benchmark results.

In [ ]:
print("\n" + "=" * 70)
print("TATR + LoRA FINE-TUNING — FINAL SUMMARY")
print("=" * 70)

print(f"\n--- Architecture ---")
print(f"Base model:          {TSR_MODEL_NAME}")
print(f"Architecture:        Vanilla TATR (DETR-style, ResNet50 + Transformer)")
print(f"Image size:          {IMAGE_SIZE}×{IMAGE_SIZE}")

print(f"\n--- LoRA Configuration ---")
print(f"Rank (r):            {LORA_R}")
print(f"Alpha:               {LORA_ALPHA}")
print(f"Dropout:             {LORA_DROPOUT}")
print(f"Bias:                none")
total_params = sum(p.numel() for p in lora_model.parameters())
trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print(f"Total params:        {total_params:,}")
print(f"Trainable params:    {trainable:,} ({100*trainable/total_params:.2f}%)")

print(f"\n--- Training ---")
print(f"Optimizer:           AdamW (lr={LEARNING_RATE_LORA}, wd={WEIGHT_DECAY})")
print(f"Scheduler:           CosineAnnealingLR (T_max={MAX_EPOCHS})")
print(f"Grad accumulation:   {GRADIENT_ACCUMULATION_STEPS} (effective batch={BATCH_SIZE*GRADIENT_ACCUMULATION_STEPS})")
print(f"Max grad norm:       {MAX_GRAD_NORM}")
print(f"Epochs trained:      {len(training_history['train_loss'])} / {MAX_EPOCHS}")
print(f"Best epoch:          {training_history['best_epoch']}")
print(f"Best val F1:         {training_history['best_val_f1']:.4f}")
print(f"AMP (fp16):          {'Yes' if DEVICE == 'cuda' else 'No'}")

print(f"\n--- Data ---")
print(f"Train samples:       {len(train_dataset)}")
print(f"Val samples:         {len(val_dataset)}")
print(f"Test samples:        {len(test_dataset)}")
print(f"Augmentation:        15 transforms (geometric + photometric + document-specific)")
print(f"Data source:         Raw COCO JSON + runtime table cropping ({TABLE_PADDING}px padding)")
print(f"Split:               Image-level ({TRAIN_RATIO:.0%}/{VAL_RATIO:.0%}/{TEST_RATIO:.0%})")

print(f"\n--- Benchmark Results ---")
for key, label in [
    ("A_Vanilla_TATR", "Vanilla TATR  "),
    ("B_LoRA_Trained", "TATR + LoRA   "),
]:
    if key in benchmark_results:
        r = benchmark_results[key]["results_50"]
        mAP = benchmark_results[key]["mAP_50_95"]
        print(f"  {label} | F1@50: {r['f1_score']:.4f} | P: {r['precision']:.4f} | R: {r['recall']:.4f} | mAP@50-95: {mAP:.4f}")

# Improvement calculation
if "A_Vanilla_TATR" in benchmark_results and "B_LoRA_Trained" in benchmark_results:
    vanilla_f1 = benchmark_results["A_Vanilla_TATR"]["results_50"]["f1_score"]
    ours_f1 = benchmark_results["B_LoRA_Trained"]["results_50"]["f1_score"]
    delta = ours_f1 - vanilla_f1
    pct = 100 * delta / vanilla_f1 if vanilla_f1 > 0 else 0
    print(f"\n  >>> F1 improvement over vanilla: {delta:+.4f} ({pct:+.1f}%)")

print(f"\n--- Output Files ---")
print(f"Best checkpoint:     {training_history.get('best_checkpoint', 'N/A')}")
print(f"Benchmark JSON:      {os.path.join(OUTPUT_DIR, 'lora_benchmark_results.json')}")
print(f"Training history:    {os.path.join(OUTPUT_DIR, 'training_history.json')}")
print(f"Training curves:     {os.path.join(OUTPUT_DIR, 'training_curves.png')}")
print(f"Benchmark chart:     {os.path.join(OUTPUT_DIR, 'benchmark_comparison.png')}")
print(f"Per-category chart:  {os.path.join(OUTPUT_DIR, 'per_category_benchmark.png')}")
print(f"Error details:       {os.path.join(OUTPUT_DIR, 'tsr_lora_error_details.csv')}")

print("\n" + "=" * 70)
print("EXPERIMENT COMPLETE")
print("=" * 70)

In [ ]:
# ================================================================
# SIDE-BY-SIDE: Vanilla vs LoRA predictions on same test samples
# ================================================================
print("=" * 70)
print("SIDE-BY-SIDE: Vanilla TATR vs TATR + LoRA — Same Test Samples")
print("=" * 70)

if "A_Vanilla_TATR" in benchmark_results and "B_LoRA_Trained" in benchmark_results:
    vanilla_preds = benchmark_results["A_Vanilla_TATR"]["predictions"]
    lora_preds = benchmark_results["B_LoRA_Trained"]["predictions"]
    
    num_compare = min(6, len(vanilla_preds), len(lora_preds))
    # Pick samples spread across the dataset
    compare_indices = np.linspace(0, min(len(vanilla_preds), len(lora_preds)) - 1, num_compare, dtype=int)
    
    colors = {2: 'blue', 3: 'green', 4: 'orange', 5: 'purple', 6: 'red'}
    
    for idx in compare_indices:
        vp = vanilla_preds[idx]
        lp = lora_preds[idx]
        
        fig, axes = plt.subplots(1, 3, figsize=(21, 7))
        
        # We don't have the actual image in memory, so reconstruct from test dataset
        try:
            pv_tensor, target = test_dataset[idx]
            img_np = pv_tensor.permute(1, 2, 0).numpy()
            img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
            img_np = np.clip(img_np, 0, 1)
        except Exception:
            img_np = np.ones((IMAGE_SIZE, IMAGE_SIZE, 3)) * 0.9
        
        h, w = img_np.shape[:2]
        
        # Ground truth
        axes[0].imshow(img_np)
        axes[0].set_title(f"Ground Truth ({len(vp['gt_boxes'])} boxes)", fontsize=12)
        for box, label in zip(vp["gt_boxes"], vp["gt_labels"]):
            x, y, bw, bh = box
            color = colors.get(label, 'gray')
            rect = patches.Rectangle((x, y), bw, bh, linewidth=2, edgecolor=color, facecolor='none', linestyle='--')
            axes[0].add_patch(rect)
        axes[0].axis("off")
        
        # Vanilla predictions
        axes[1].imshow(img_np)
        axes[1].set_title(f"Vanilla TATR ({len(vp['pred_boxes'])} preds)", fontsize=12, color="blue")
        for box, score, label in zip(vp["pred_boxes"], vp["pred_scores"], vp["pred_labels"]):
            x, y, bw, bh = box
            color = colors.get(label, 'gray')
            rect = patches.Rectangle((x, y), bw, bh, linewidth=2, edgecolor=color, facecolor='none')
            axes[1].add_patch(rect)
        axes[1].axis("off")
        
        # LoRA predictions
        axes[2].imshow(img_np)
        axes[2].set_title(f"TATR + LoRA ({len(lp['pred_boxes'])} preds)", fontsize=12, color="green")
        for box, score, label in zip(lp["pred_boxes"], lp["pred_scores"], lp["pred_labels"]):
            x, y, bw, bh = box
            color = colors.get(label, 'gray')
            rect = patches.Rectangle((x, y), bw, bh, linewidth=2, edgecolor=color, facecolor='none')
            axes[2].add_patch(rect)
        axes[2].axis("off")
        
        # Compute per-sample F1 for both
        tp_v, fp_v, fn_v, ious_v, _, _ = match_predictions_to_gt(
            vp["pred_boxes"], vp["pred_scores"], vp["pred_labels"],
            vp["gt_boxes"], vp["gt_labels"], 0.5)
        tp_l, fp_l, fn_l, ious_l, _, _ = match_predictions_to_gt(
            lp["pred_boxes"], lp["pred_scores"], lp["pred_labels"],
            lp["gt_boxes"], lp["gt_labels"], 0.5)
        
        f1_v = 2*tp_v / (2*tp_v + fp_v + fn_v) if (2*tp_v + fp_v + fn_v) > 0 else 0
        f1_l = 2*tp_l / (2*tp_l + fp_l + fn_l) if (2*tp_l + fp_l + fn_l) > 0 else 0
        
        plt.suptitle(f"Sample {idx} — Vanilla F1: {f1_v:.3f} vs LoRA F1: {f1_l:.3f}", 
                     fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.show()
else:
    print("Need both Vanilla and LoRA benchmark results for comparison.")

print("\nNotebook execution complete!")